# Cold-Start Benchmark and Architectural Diagnosis for RPI Prediction

This notebook contains the full pipeline that produces every table and
figure in the paper. It is organised as follows:

1. **Setup** — imports, device, data paths.
2. **Library** — data loading, metrics, graph construction, model
   definitions, training loop, and split builders.
3. **Diagnostics** — degree-distribution sanity checks.
4. **Within-dataset ablation** → Tables I, II, III.
5. **Cross-dataset (TheNovel) evaluation** → Table V.
6. **GraphSAGE tuning sweep on TheNovel** → Table VI.
7. **Case studies** → Figures 1, 2, 3.

The library sections (1–2) define everything
the later experiment sections call into. The experiment sections can be
re-run independently once the library is loaded.

Datasets and embeddings are the ZHMolGraph-released benchmarks; see
`data/README.md` for the Zenodo link and the expected folder layout.


## 1. Setup

In [ ]:
import os, math, random, pickle as pkl
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             confusion_matrix)
from scipy import stats as scipy_stats       
from collections import Counter
import itertools
from functools import partial

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_all(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE   = "/kaggle/input/datasets/abdullahnayem/dataset"   # ← change accordingly
print("DEVICE:", DEVICE)
print("BASE exists:", os.path.exists(BASE))


## 2. Library

### 2.1 Data loading

`build_graph_data(name)` returns `(x_rna, x_prot, r_idx, p_idx, y, meta)`
for `name ∈ {"NPInter2", "RPI7317"}`. Embeddings are frozen RNA-FM
(640-d) and ProtT5 (1024-d) reused from ZHMolGraph; identifiers are
mapped to unique sequences.

In [ ]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"None of {candidates} found. Got: {list(df.columns)}")

def load_embed_pkls(dataset_name, base_dir=BASE):
    rna_pkl  = os.path.join(base_dir, f"RPI_{dataset_name}_rnafm_embed_normal.pkl")
    prot_pkl = os.path.join(base_dir, f"RPI_{dataset_name}_proteinprottrans_embed_normal.pkl")
    with open(rna_pkl,  "rb") as f: rna_df  = pkl.load(f)
    with open(prot_pkl, "rb") as f: prot_df = pkl.load(f)
    rna_seq_col  = pick_col(rna_df,  ["RNA_aa_code",    "RNA_seq",     "rna_seq"])
    prot_seq_col = pick_col(prot_df, ["target_aa_code", "protein_seq", "prot_seq"])
    emb_col      = pick_col(rna_df,  ["normalized_embeddings", "embeddings", "embed"])
    return rna_df, prot_df, rna_seq_col, prot_seq_col, emb_col

def load_interactions_csv(dataset_name, base_dir=BASE):
    csv_path = os.path.join(base_dir, f"dataset_RPI_{dataset_name}_RP.csv")
    df = pd.read_csv(csv_path)
    rna_col  = pick_col(df, ["RNA_aa_code", "RNA",     "rna",     "rna_seq"])
    prot_col = pick_col(df, ["target_aa_code", "Protein", "protein", "prot_seq"])
    y_col    = pick_col(df, ["Y", "label", "Label", "y"])
    df[y_col] = df[y_col].astype(int)
    return df, rna_col, prot_col, y_col

def build_graph_data(dataset_name, base_dir=BASE):
    """Returns x_rna, x_prot, r_idx, p_idx, y, meta."""
    rna_df, prot_df, rna_seq_col, prot_seq_col, emb_col = load_embed_pkls(dataset_name, base_dir)
    inter_df, rna_col, prot_col, y_col = load_interactions_csv(dataset_name, base_dir)

    rna_seqs  = rna_df[rna_seq_col].astype(str).tolist()
    prot_seqs = prot_df[prot_seq_col].astype(str).tolist()
    rna_map   = {s: i for i, s in enumerate(rna_seqs)}
    prot_map  = {s: i for i, s in enumerate(prot_seqs)}

    x_rna  = np.stack(rna_df[emb_col].to_list()).astype(np.float32)
    x_prot = np.stack(prot_df[emb_col].to_list()).astype(np.float32)

    r_idx = inter_df[rna_col].astype(str).map(rna_map)
    p_idx = inter_df[prot_col].astype(str).map(prot_map)
    mask  = r_idx.notna() & p_idx.notna()

    inter_df2 = inter_df.loc[mask].reset_index(drop=True)
    r_idx = r_idx.loc[mask].astype(int).to_numpy()
    p_idx = p_idx.loc[mask].astype(int).to_numpy()
    y     = inter_df2[y_col].astype(int).to_numpy()

    meta = dict(
        inter_df=inter_df2, rna_col=rna_col, prot_col=prot_col, y_col=y_col,
        n_rna=x_rna.shape[0], n_prot=x_prot.shape[0],
        d_rna=x_rna.shape[1], d_prot=x_prot.shape[1],
    )
    print(f"[{dataset_name}] RNAs={meta['n_rna']} Proteins={meta['n_prot']} "
          f"Edges={len(y)} Pos={y.sum()} Neg={(y==0).sum()}")
    return x_rna, x_prot, r_idx, p_idx, y, meta


### 2.2 Metrics

Threshold-free metrics (AUROC, AUPRC) and MCC at a tuned threshold.
`best_thr_by_mcc` scans a 200-point grid on the inner validation set.

In [ ]:
def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    pos = x >= 0; neg = ~pos
    out[pos] = 1.0 / (1.0 + np.exp(-x[pos]))
    ex = np.exp(x[neg]); out[neg] = ex / (1.0 + ex)
    return out.astype(np.float32)

def confusion_counts(y_true, y_pred):
    y_true = y_true.astype(int); y_pred = y_pred.astype(int)
    tp = int(((y_true==1)&(y_pred==1)).sum()); tn = int(((y_true==0)&(y_pred==0)).sum())
    fp = int(((y_true==0)&(y_pred==1)).sum()); fn = int(((y_true==1)&(y_pred==0)).sum())
    return tp, tn, fp, fn

def mcc_score(tp, tn, fp, fn):
    num = tp*tn - fp*fn
    den = math.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn) + 1e-12)
    return float(num/den) if den > 0 else 0.0

def safe_auroc_auprc(y_true, y_score):
    try:    auroc = float(roc_auc_score(y_true, y_score))
    except: auroc = float("nan")
    try:    auprc = float(average_precision_score(y_true, y_score))
    except: auprc = float("nan")
    return auroc, auprc

def compute_metrics_from_logits(y_true, y_logits, thr=0.5):
    y_prob = sigmoid_np(y_logits)
    y_pred = (y_prob >= thr).astype(int)
    tp, tn, fp, fn = confusion_counts(y_true, y_pred)
    auroc, auprc = safe_auroc_auprc(y_true, y_prob)
    return dict(
        ACC=(tp+tn)/max(tp+tn+fp+fn,1),
        SEN=tp/max(tp+fn,1), SPE=tn/max(tn+fp,1), PRE=tp/max(tp+fp,1),
        MCC=mcc_score(tp,tn,fp,fn), AUROC=auroc, AUPRC=auprc
    )

def best_thr_by_mcc(y_true, y_logits, n_grid=200):
    """Scan threshold grid on VAL logits; return (best_thr, best_mcc)."""
    y_prob = sigmoid_np(np.asarray(y_logits))
    best_t, best_m = 0.5, -1.0
    for t in np.linspace(0.0, 1.0, n_grid + 1):
        tp, tn, fp, fn = confusion_counts(y_true, (y_prob >= t).astype(int))
        m = mcc_score(tp, tn, fp, fn)
        if m > best_m:
            best_m, best_t = m, float(t)
    return best_t, best_m


### 2.3 Graph construction

Bipartite training graph built strictly from training positives (no
label leakage). `symm_norm_msgs` implements LightGCN-style symmetric
normalization.

In [ ]:
def build_train_graph_from_split(r_idx, p_idx, y, train_row_idx, n_rna, n_prot):
    """Sparse bipartite adjacency from TRAIN POSITIVES only (no leakage)."""
    tr = train_row_idx
    pos_mask = (y[tr] == 1)
    r_pos = r_idx[tr][pos_mask]
    p_pos = p_idx[tr][pos_mask]
    if len(r_pos) == 0:
        raise ValueError("No positive edges in train split.")
    ij = np.vstack([r_pos, p_pos])
    vals = np.ones(ij.shape[1], dtype=np.float32)
    A_rp = torch.sparse_coo_tensor(
        torch.tensor(ij, dtype=torch.long),
        torch.tensor(vals),
        size=(n_rna, n_prot)
    ).coalesce()
    rna2prot = [[] for _ in range(n_rna)]
    prot2rna = [[] for _ in range(n_prot)]
    for rr, pp in zip(r_pos.tolist(), p_pos.tolist()):
        rna2prot[rr].append(pp); prot2rna[pp].append(rr)
    return A_rp, rna2prot, prot2rna

def sparse_edge_dropout(A, p, training):
    if (not training) or p <= 0.0: return A
    A = A.coalesce()
    val = A.values()
    keep = (torch.rand_like(val) > p).to(val.dtype)
    val2 = val * keep / (1.0 - p + 1e-12)
    return torch.sparse_coo_tensor(A.indices(), val2, A.size(), device=A.device).coalesce()

def symm_norm_msgs(A, h_rna, h_prot):
    """LightGCN-style symmetric normalization on bipartite graph."""
    A = A.coalesce()
    deg_r = torch.sparse.sum(A, dim=1).to_dense().clamp_min(1.0)
    deg_p = torch.sparse.sum(A, dim=0).to_dense().clamp_min(1.0)
    inv_sqrt_r = deg_r.rsqrt(); inv_sqrt_p = deg_p.rsqrt()
    h_rna_n  = h_rna  * inv_sqrt_r.unsqueeze(-1)
    h_prot_n = h_prot * inv_sqrt_p.unsqueeze(-1)
    rna_msg  = torch.sparse.mm(A,               h_prot_n) * inv_sqrt_r.unsqueeze(-1)
    prot_msg = torch.sparse.mm(A.transpose(0,1), h_rna_n) * inv_sqrt_p.unsqueeze(-1)
    return rna_msg, prot_msg, deg_r, deg_p


### 2.4 Models

All models share the `encode_nodes` / `score_edges` interface used by the
CV runners.

* PairMLP family (no graph): base, Wide, Deep, Cross-attention.
* LightGCN-style: DegGate, WeightedBipartite, CoNeighbor, ProtoContrast.
* GraphSAGE-style: GraphSAGE-2L (2-layer, k=25 neighbours by default).

In [ ]:
class PairMLP(nn.Module):
    """
     PairMLP baseline:
      1. enc_rna : d_rna → d   (Linear → ReLU → Dropout)
      2. enc_prot: d_prot → d  (Linear → ReLU → Dropout)
      3. interaction vector z = [hr; hp; |hr-hp|; hr⊙hp]  ∈ R^{4d}
      4. MLP head: 4d → 512 → 256 → 1

    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))
        self.mlp = nn.Sequential(
            nn.Linear(4*d, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        """x_rna_batch: (B, d_rna),  x_prot_batch: (B, d_prot) → logits (B,)"""
        hr = self.enc_rna(x_rna_batch)
        hp = self.enc_prot(x_prot_batch)
        z  = torch.cat([hr, hp, torch.abs(hr-hp), hr*hp], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLPGraphWrapper(nn.Module):
    """
    Wraps PairMLP to expose the encode_nodes / score_edges interface
    used by all CV runners and the retrieval pipeline.
    PairMLP doesn't use the graph — the wrapper just stores x_rna/x_prot
    so score_edges can look them up.
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP(d_rna, d_prot, d, dropout)
        self._x_rna  = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        """Store tensors; return dummies so the interface is satisfied."""
        self._x_rna  = x_rna
        self._x_prot = x_prot
        dummy_core = torch.zeros(1, device=x_rna.device)
        return x_rna, x_prot, dummy_core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])


In [ ]:
"""
Three variants:
  1. PairMLP_Wide       : no projection, full 1664-dim concatenated embeddings,
                          deeper MLP head
  2. PairMLP_Deep       : current 256-dim projection, 5-layer MLP head
  3. PairMLP_Cross      : small cross-attention between RNA and protein
                          before the standard 4-feature interaction

All three use the same wrapper pattern as our existing PairMLPGraphWrapper.
"""



class PairMLP_Wide(nn.Module):
    """
    No projection. Uses raw concatenated embeddings.
      - z = [h_rna ; h_prot ; |h_rna - h_prot_proj| ; h_rna_proj * h_prot_proj]
      - But |diff| and product require same dim, so we project ONLY for those
        two features and keep the raw embeddings as-is for the concat.
    Final input dim: d_rna + d_prot + d + d = 640 + 1024 + 256 + 256 = 2176
    MLP head: 2176 -> 1024 -> 512 -> 256 -> 1
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        # Small projections JUST for the diff and product features
        self.proj_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU())
        self.proj_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU())

        in_dim = d_rna + d_prot + d + d
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024,    512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,     256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,       1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        pr = self.proj_rna(x_rna_batch)    # (B, d)
        pp = self.proj_prot(x_prot_batch)  # (B, d)
        z = torch.cat([
            x_rna_batch,         # raw RNA  (B, 640)
            x_prot_batch,        # raw prot (B, 1024)
            torch.abs(pr - pp),  # diff in projected space (B, 256)
            pr * pp,             # product in projected space (B, 256)
        ], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLP_Wide_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP_Wide(d_rna, d_prot, d, dropout)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])



class PairMLP_Deep(nn.Module):
    """
    Same input as base PairMLP (4d=1024-dim interaction vector after projection).
    Deeper head with residual blocks:
      4d -> 768 -> 512 -> 384 -> 256 -> 1
    Plus a residual skip from layer 1 to layer 4.
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))

        self.layer1 = nn.Sequential(nn.Linear(4*d, 768), nn.ReLU(), nn.Dropout(dropout))
        self.layer2 = nn.Sequential(nn.Linear(768,  512), nn.ReLU(), nn.Dropout(dropout))
        self.layer3 = nn.Sequential(nn.Linear(512,  384), nn.ReLU(), nn.Dropout(dropout))
        self.layer4 = nn.Sequential(nn.Linear(384,  256), nn.ReLU(), nn.Dropout(dropout))

        # Residual projection from layer1 output (768) down to layer4 output (256)
        self.res_proj = nn.Linear(768, 256, bias=False)

        self.head = nn.Linear(256, 1)

    def forward(self, x_rna_batch, x_prot_batch):
        hr = self.enc_rna(x_rna_batch)
        hp = self.enc_prot(x_prot_batch)
        z = torch.cat([hr, hp, torch.abs(hr - hp), hr * hp], dim=1)  # (B, 4d)

        a1 = self.layer1(z)             # (B, 768)
        a2 = self.layer2(a1)            # (B, 512)
        a3 = self.layer3(a2)            # (B, 384)
        a4 = self.layer4(a3)            # (B, 256)
        a4 = a4 + self.res_proj(a1)     # residual from layer1 -> layer4

        return self.head(a4).squeeze(-1)

class PairMLP_Deep_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP_Deep(d_rna, d_prot, d, dropout)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])



class PairMLP_Cross(nn.Module):
    """
    Cross-attention between RNA and protein, then standard 4-feature MLP head.
      hr, hp = enc_rna(x_rna), enc_prot(x_prot)         # (B, d) each
      hr_new = hr + Attn(query=hr, key=hp, value=hp)
      hp_new = hp + Attn(query=hp, key=hr, value=hr)
      z = [hr_new ; hp_new ; |hr_new - hp_new| ; hr_new * hp_new]   # (B, 4d)
      logit = MLP(z)
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2, n_heads=4):
        super().__init__()
        assert d % n_heads == 0, f"d={d} must be divisible by n_heads={n_heads}"

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))

        # Two cross-attention modules: RNA queries protein, protein queries RNA
        self.attn_r2p = nn.MultiheadAttention(d, num_heads=n_heads,
                                              dropout=dropout, batch_first=True)
        self.attn_p2r = nn.MultiheadAttention(d, num_heads=n_heads,
                                              dropout=dropout, batch_first=True)

        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)

        self.mlp = nn.Sequential(
            nn.Linear(4*d, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        hr = self.enc_rna(x_rna_batch)    # (B, d)
        hp = self.enc_prot(x_prot_batch)  # (B, d)

        # Reshape to (B, seq_len=1, d) for MultiheadAttention with batch_first=True
        hr_seq = hr.unsqueeze(1)
        hp_seq = hp.unsqueeze(1)

        # Cross-attention
        hr_attn, _ = self.attn_r2p(query=hr_seq, key=hp_seq, value=hp_seq)
        hp_attn, _ = self.attn_p2r(query=hp_seq, key=hr_seq, value=hr_seq)

        # Residual + norm, then squeeze back to (B, d)
        hr_new = self.norm_r(hr + hr_attn.squeeze(1))
        hp_new = self.norm_p(hp + hp_attn.squeeze(1))

        # Standard 4-feature interaction
        z = torch.cat([hr_new, hp_new,
                       torch.abs(hr_new - hp_new),
                       hr_new * hp_new], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLP_Cross_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2, n_heads=4):
        super().__init__()
        self.model = PairMLP_Cross(d_rna, d_prot, d, dropout, n_heads)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])


In [ ]:
# 1. ProtoGNN_DegGate
#    GNN + degree-gated fusion + pair gate + global core (TypeAwareCoreFusion).
#    NO prototypes, NO contrastive term.

class TypeAwareCoreFusion(nn.Module):
    """
    Separate attention pooling for RNA and Protein, then fuse.
    Returns one (d,) core vector so your train/eval code remains unchanged.
    """
    def __init__(self, d):
        super().__init__()
        self.att_r = nn.Linear(d, 1)
        self.att_p = nn.Linear(d, 1)
        self.fuse  = nn.Sequential(
            nn.Linear(2*d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

    def forward(self, h_rna, h_prot):
        wr = torch.softmax(self.att_r(h_rna).squeeze(-1), dim=0)  # (n_rna,)
        wp = torch.softmax(self.att_p(h_prot).squeeze(-1), dim=0) # (n_prot,)
        cr = (wr.unsqueeze(-1) * h_rna).sum(dim=0)
        cp = (wp.unsqueeze(-1) * h_prot).sum(dim=0)
        return self.fuse(torch.cat([cr, cp], dim=0))

class ProtoGNN_DegGate(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        # Encoders: project raw LLM embeddings to shared d-dim space
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        # LightGCN-style message passing with post-aggregation mixing
        self.mix_r = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(),
                                    nn.Dropout(dropout)) for _ in range(layers)])
        self.mix_p = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(),
                                    nn.Dropout(dropout)) for _ in range(layers)])
        self.norm_r = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.norm_p = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])

        # Degree gate: g = sigmoid(MLP(log(1+deg))), blends raw ↔ graph
        self.deg_gate_r = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, 1))
        self.deg_gate_p = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, 1))

        # Global core vector (attention-pooled, NOT per-node prototypes)
        self.core = TypeAwareCoreFusion(d)

        # Per-edge pair gate: decides how much to trust graph vs raw per edge
        self.pair_gate = nn.Sequential(
            nn.Linear(d*4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1)
        )
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, hr0, hp0, core] → logit
        in_edge = (d*4) + (d*2) + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        hg_r, hg_p = h0_r, h0_p

        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for i in range(self.layers):
            r_msg, p_msg, _, _ = symm_norm_msgs(A, hg_r, hg_p)
            hg_r = self.norm_r[i](hg_r + self.mix_r[i](r_msg))
            hg_p = self.norm_p[i](hg_p + self.mix_p[i](p_msg))

        # Degree from ORIGINAL adjacency (not edge-dropped)
        A0 = A_rp.coalesce()
        deg_r = torch.sparse.sum(A0, dim=1).to_dense().clamp_min(0.0)
        deg_p = torch.sparse.sum(A0, dim=0).to_dense().clamp_min(0.0)

        # Cold-node fix: hard-zero gate for degree-0 nodes → pure raw embedding
        is_cold_r = (deg_r == 0).float().unsqueeze(-1)
        is_cold_p = (deg_p == 0).float().unsqueeze(-1)
        g_r = torch.sigmoid(self.deg_gate_r(torch.log1p(deg_r).unsqueeze(-1)))
        g_p = torch.sigmoid(self.deg_gate_p(torch.log1p(deg_p).unsqueeze(-1)))
        h_r = h0_r + (1.0 - is_cold_r) * g_r * (hg_r - h0_r)
        h_p = h0_p + (1.0 - is_cold_p) * g_p * (hg_p - h0_p)

        # Global core from ALL nodes (single vector, no per-node prototypes)
        # Note: this includes cold nodes, but it's just attention-pooled —
        # not a per-node soft assignment like TypeAwareCorePrototypes.
        core = self.core(h_r, h_p)

        self._cache_h0 = (h0_r, h0_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]
        hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids]
        hp0 = h0_p[p_ids]

        # Per-edge gate: blend raw → graph
        a = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0 - hp0), hr0 * hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)

        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            hr0, hp0,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits

# 2. ProtoGNN_WeightedBipartite
#    Attention-weighted message passing (BipartiteWeightConv).
#    NO degree gate, NO pair gate, NO prototypes.
#    Cold nodes: no messages arrive (degree 0), so conv output = raw encoded
#    embedding (residual connection in BipartiteWeightConv handles this since
#    out_r = 0 for degree-0 nodes, and h_r2 = norm(h_rna + act(0))).

class BipartiteWeightConv(nn.Module):
    """
    Attention-weighted bipartite message passing.
    w = sigmoid(<Wq u, Wk v> / sqrt(d)), normalized by sum_w per node.
    Cold nodes (degree 0): no edges → out = zeros → residual keeps raw embedding.
    """
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.Wq_r = nn.Linear(d, d, bias=False)
        self.Wk_p = nn.Linear(d, d, bias=False)
        self.Wv_p = nn.Linear(d, d, bias=False)
        self.Wq_p = nn.Linear(d, d, bias=False)
        self.Wk_r = nn.Linear(d, d, bias=False)
        self.Wv_r = nn.Linear(d, d, bias=False)
        self.drop  = nn.Dropout(dropout)
        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)
        self.act = nn.GELU()

    def forward(self, h_rna, h_prot, A_rp):
        A = A_rp.coalesce()
        idx = A.indices()
        r = idx[0]   # (E,)
        p = idx[1]   # (E,)
        n_rna  = h_rna.size(0)
        n_prot = h_prot.size(0)
        d = h_rna.size(1)
        scale = 1.0 / math.sqrt(d)

        # RNA ← Protein messages (attention-weighted)
        q = self.Wq_r(h_rna[r])
        k = self.Wk_p(h_prot[p])
        w = torch.sigmoid((q * k).sum(-1) * scale)
        v = self.drop(self.Wv_p(h_prot[p])) * w.unsqueeze(-1)
        out_r = torch.zeros(n_rna, d, device=h_rna.device)
        out_r.index_add_(0, r, v)
        den_r = torch.zeros(n_rna, device=h_rna.device)
        den_r.index_add_(0, r, w)
        out_r = out_r / den_r.clamp_min(1e-6).unsqueeze(-1)

        # Protein ← RNA messages (attention-weighted)
        q2 = self.Wq_p(h_prot[p])
        k2 = self.Wk_r(h_rna[r])
        w2 = torch.sigmoid((q2 * k2).sum(-1) * scale)
        v2 = self.drop(self.Wv_r(h_rna[r])) * w2.unsqueeze(-1)
        out_p = torch.zeros(n_prot, d, device=h_rna.device)
        out_p.index_add_(0, p, v2)
        den_p = torch.zeros(n_prot, device=h_rna.device)
        den_p.index_add_(0, p, w2)
        out_p = out_p / den_p.clamp_min(1e-6).unsqueeze(-1)

        # Residual + norm (cold nodes: out=0, so h_r2 ≈ norm(h_rna + act(0)))
        h_r2 = self.norm_r(h_rna + self.act(out_r))
        h_p2 = self.norm_p(h_prot + self.act(out_p))
        return h_r2, h_p2

class ProtoGNN_WeightedBipartite(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))
        self.convs = nn.ModuleList([BipartiteWeightConv(d, dropout=dropout)
                                    for _ in range(layers)])

        # Global core (single vector, no prototypes)
        self.core  = TypeAwareCoreFusion(d)
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, core] → logit
        in_edge = d*4 + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h_r = self.enc_rna(x_rna)
        h_p = self.enc_prot(x_prot)
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for conv in self.convs:
            h_r, h_p = conv(h_r, h_p, A)
        core = self.core(h_r, h_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr = h_rna[r_ids]
        hp = h_prot[p_ids]
        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits

# 3. ProtoGNN_CoNeighbor
#    1-hop + 2-hop (co-neighbor) propagation using symm_norm_msgs.
#    NO degree gate, NO prototypes.
#    Cold nodes: 1-hop messages = 0 (no neighbours), 2-hop also = 0.
#    Residual in CoNeighborLayer keeps raw encoded embedding.

class CoNeighborLayer(nn.Module):
    """
    1-hop + 2-hop (co-neighbor) propagation on bipartite graph.
      1-hop: RNA ← Prot, Prot ← RNA  (standard LightGCN)
      2-hop: RNA ← Prot ← RNA,  Prot ← RNA ← Prot  (co-neighbor = same-type similarity)
    Cold nodes: both hops produce zero messages → residual preserves input.
    """
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.mlp_r = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout))
        self.mlp_p = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout))
        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)

    def forward(self, h_r, h_p, A):
        # 1-hop: standard bipartite message passing
        r1, p1, _, _ = symm_norm_msgs(A, h_r, h_p)
        # 2-hop: co-neighbor (same-type indirect similarity through opposite type)
        # RNA gets messages from proteins who already aggregated from RNAs
        r2, _, _, _ = symm_norm_msgs(A, h_r, p1)
        # Protein gets messages from RNAs who already aggregated from proteins
        _, p2, _, _ = symm_norm_msgs(A, r1, h_p)

        h_r2 = self.norm_r(h_r + self.mlp_r(torch.cat([r1, r2], dim=1)))
        h_p2 = self.norm_p(h_p + self.mlp_p(torch.cat([p1, p2], dim=1)))
        return h_r2, h_p2

class ProtoGNN_CoNeighbor(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))
        self.blocks = nn.ModuleList([CoNeighborLayer(d, dropout=dropout)
                                     for _ in range(layers)])

        # Global core (single vector, no prototypes)
        self.core  = TypeAwareCoreFusion(d)
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, core] → logit
        in_edge = d*4 + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h_r = self.enc_rna(x_rna)
        h_p = self.enc_prot(x_prot)
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for blk in self.blocks:
            h_r, h_p = blk(h_r, h_p, A)
        core = self.core(h_r, h_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr = h_rna[r_ids]
        hp = h_prot[p_ids]
        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits


In [ ]:
def _l2norm(x, eps=1e-12):
    return x / (x.norm(dim=-1, keepdim=True).clamp_min(eps))

class TypeAwareCorePrototypes(nn.Module):
    def __init__(self, d, K=4, dropout=0.1):
        super().__init__()
        self.K = K
        self.key_r = nn.Linear(d, d, bias=False)
        self.key_p = nn.Linear(d, d, bias=False)
        self.q_r   = nn.Parameter(torch.randn(K, d) * 0.02)
        self.q_p   = nn.Parameter(torch.randn(K, d) * 0.02)
        self.drop  = nn.Dropout(dropout)
        self.core_out = nn.Sequential(
            nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, h_r, h_p):
        # h_r: (N_warm_rna, d)  h_p: (N_warm_prot, d)
        Kr = self.key_r(self.drop(h_r))
        Kp = self.key_p(self.drop(h_p))
        ar = torch.softmax(Kr @ self.q_r.t(), dim=0)  # (N_warm_rna, K)
        ap = torch.softmax(Kp @ self.q_p.t(), dim=0)  # (N_warm_prot, K)
        proto_r = ar.t() @ h_r   # (K, d)
        proto_p = ap.t() @ h_p   # (K, d)
        core_global = self.core_out(
            torch.cat([proto_r.mean(0), proto_p.mean(0)], dim=0)
        )
        return proto_r, proto_p, core_global

# 4. ProtoGNN_ProtoContrast

class ProtoGNN_ProtoContrast(nn.Module):
  
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 edge_drop=0.0, K=4, tau=0.2, contrast_w=0.2):
        super().__init__()
        self.layers = layers; self.edge_drop = edge_drop
        self.K = K; self.tau = tau; self.contrast_w = contrast_w

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        self.mix_r  = nn.ModuleList([nn.Sequential(nn.Linear(d,d), nn.GELU(),
                                     nn.Dropout(dropout)) for _ in range(layers)])
        self.mix_p  = nn.ModuleList([nn.Sequential(nn.Linear(d,d), nn.GELU(),
                                     nn.Dropout(dropout)) for _ in range(layers)])
        self.norm_r = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.norm_p = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])

        self.deg_gate_r = nn.Sequential(nn.Linear(1,32), nn.GELU(), nn.Linear(32,1))
        self.deg_gate_p = nn.Sequential(nn.Linear(1,32), nn.GELU(), nn.Linear(32,1))

        self.pair_gate = nn.Sequential(
            nn.Linear(d*4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256,1)
        )
        self.proto_core     = TypeAwareCorePrototypes(d, K=K, dropout=dropout)
        self.edge_core_q    = nn.Linear(d, d, bias=False)
        self.edge_core_fuse = nn.Sequential(nn.Linear(2*d,d), nn.GELU(), nn.Dropout(dropout))
        self.bilin          = nn.Bilinear(d, d, 1, bias=False)

        in_edge = (d*4) + (d*2) + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    # ── encode ──────────────────────────────────────────────────────────────
    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        hg_r, hg_p = h0_r, h0_p
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)

        for i in range(self.layers):
            r_msg, p_msg, _, _ = symm_norm_msgs(A, hg_r, hg_p)
            hg_r = self.norm_r[i](hg_r + self.mix_r[i](r_msg))
            hg_p = self.norm_p[i](hg_p + self.mix_p[i](p_msg))

        # degree from ORIGINAL adjacency (not dropped)
        A0    = A_rp.coalesce()
        deg_r = torch.sparse.sum(A0, dim=1).to_dense().clamp_min(0.0)
        deg_p = torch.sparse.sum(A0, dim=0).to_dense().clamp_min(0.0)

        # hard-zero gate for cold nodes → they keep pure raw embedding
        is_cold_r = (deg_r == 0).float().unsqueeze(-1)
        is_cold_p = (deg_p == 0).float().unsqueeze(-1)
        g_r = torch.sigmoid(self.deg_gate_r(torch.log1p(deg_r).unsqueeze(-1)))
        g_p = torch.sigmoid(self.deg_gate_p(torch.log1p(deg_p).unsqueeze(-1)))
        h_r = h0_r + (1.0 - is_cold_r) * g_r * (hg_r - h0_r)
        h_p = h0_p + (1.0 - is_cold_p) * g_p * (hg_p - h0_p)

        #  prototypes from warm (training) nodes only
        warm_r = (deg_r > 0)
        warm_p = (deg_p > 0)
        proto_r, proto_p, core_global = self.proto_core(h_r[warm_r], h_p[warm_p])

        self._cache_h0 = (h0_r, h0_p)
        self._proto_r  = proto_r   # (K, d) — warm nodes only
        self._proto_p  = proto_p
        return h_r, h_p, core_global

    # ── edge core from prototypes ────────────────────────────────────────────
    def _edge_core_from_prototypes(self, hr, hp):
        scale = 1.0 / math.sqrt(hr.size(1))
        qr = self.edge_core_q(hr)
        qp = self.edge_core_q(hp)
        ar = torch.softmax((qr @ self._proto_r.t()) * scale, dim=1)
        ap = torch.softmax((qp @ self._proto_p.t()) * scale, dim=1)
        return self.edge_core_fuse(
            torch.cat([ar @ self._proto_r, ap @ self._proto_p], dim=1)
        )

    # ── contrastive score ────────────────────────────────────────────────────
    def _contrastive_score(self, hr, hp):
        tau  = max(self.tau, 1e-6)
        hr_n = _l2norm(hr); hp_n = _l2norm(hp)
        pr   = _l2norm(self._proto_r); pp = _l2norm(self._proto_p)
        sim_pos = (hr_n * hp_n).sum(-1) / tau
        neg_p   = torch.logsumexp((hr_n @ pp.t()) / tau, dim=1)
        neg_r   = torch.logsumexp((hp_n @ pr.t()) / tau, dim=1)
        return sim_pos - 0.5 * (neg_p + neg_r)

    # ── score edges ──────────────────────────────────────────────────────────
    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]; hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids];   hp0 = h0_p[p_ids]

        a  = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0-hp0), hr0*hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)

        feat = torch.cat([
            hr, hp, torch.abs(hr-hp), hr*hp,
            hr0, hp0,
            self._edge_core_from_prototypes(hr, hp)
        ], dim=1)

        logits = (self.edge_mlp(feat).squeeze(-1)
                  + self.bilin(hr, hp).squeeze(-1)
                  + self.contrast_w * self._contrastive_score(hr, hp))
        return logits


In [ ]:
def build_neighbour_lists(A_rp, n_rna, n_prot, device):
    """A_rp: sparse (n_rna, n_prot). Returns (rna2prot_list, prot2rna_list)."""
    A = A_rp.coalesce()
    idx = A.indices()                    # (2, E)
    r_ids = idx[0]                       # (E,)
    p_ids = idx[1]                       # (E,)
 
    # Group protein indices by RNA
    rna2prot = [torch.empty(0, dtype=torch.long, device=device) for _ in range(n_rna)]
    prot2rna = [torch.empty(0, dtype=torch.long, device=device) for _ in range(n_prot)]
 
    # Vectorised grouping using sorting
    if r_ids.numel() > 0:
        # RNA -> protein
        order_r = torch.argsort(r_ids)
        r_sorted = r_ids[order_r]
        p_sorted = p_ids[order_r]
        # find boundaries where r changes
        unique_r, counts_r = torch.unique_consecutive(r_sorted, return_counts=True)
        # Split p_sorted into chunks corresponding to each unique r
        offsets_r = torch.cat([torch.zeros(1, dtype=torch.long, device=device),
                               counts_r.cumsum(0)])
        for k, r_val in enumerate(unique_r.tolist()):
            rna2prot[r_val] = p_sorted[offsets_r[k]:offsets_r[k+1]]
 
        # Protein -> RNA
        order_p = torch.argsort(p_ids)
        p_sorted2 = p_ids[order_p]
        r_sorted2 = r_ids[order_p]
        unique_p, counts_p = torch.unique_consecutive(p_sorted2, return_counts=True)
        offsets_p = torch.cat([torch.zeros(1, dtype=torch.long, device=device),
                               counts_p.cumsum(0)])
        for k, p_val in enumerate(unique_p.tolist()):
            prot2rna[p_val] = r_sorted2[offsets_p[k]:offsets_p[k+1]]
 
    return rna2prot, prot2rna

class BipartiteSAGELayer(nn.Module):
    def __init__(self, d, dropout=0.1, sample_k=25, use_l2norm=True):
        """
        sample_k: int = sample this many neighbours per node during training,
                  None = use ALL neighbours (no sampling).
        use_l2norm: bool = apply F.normalize(..., p=2) per layer.
        """
        super().__init__()
        self.d = d
        self.sample_k = sample_k
        self.use_l2norm = use_l2norm
        self.W_rna  = nn.Linear(2 * d, d)
        self.W_prot = nn.Linear(2 * d, d)
        self.dropout = nn.Dropout(dropout)

    def _sample_and_aggregate(self, h_neighbour, neighbour_lists, n_self):
        d = h_neighbour.size(1)
        device = h_neighbour.device
        agg = torch.zeros(n_self, d, device=device)

        # Use all neighbours when (a) sample_k is None or (b) we're in eval mode
        use_all = (self.sample_k is None) or (not self.training)

        if use_all:
            for i in range(n_self):
                nb = neighbour_lists[i]
                if nb.numel() > 0:
                    agg[i] = h_neighbour[nb].mean(dim=0)
        else:
            for i in range(n_self):
                nb = neighbour_lists[i]
                if nb.numel() == 0:
                    continue
                if nb.numel() <= self.sample_k:
                    chosen = nb
                else:
                    perm = torch.randperm(nb.numel(), device=device)[:self.sample_k]
                    chosen = nb[perm]
                agg[i] = h_neighbour[chosen].mean(dim=0)
        return agg

    def forward(self, h_rna, h_prot, rna2prot, prot2rna):
        agg_for_rna  = self._sample_and_aggregate(h_prot, rna2prot, h_rna.size(0))
        agg_for_prot = self._sample_and_aggregate(h_rna, prot2rna, h_prot.size(0))

        h_rna_new  = F.relu(self.W_rna( self.dropout(torch.cat([h_rna,  agg_for_rna],  dim=1))))
        h_prot_new = F.relu(self.W_prot(self.dropout(torch.cat([h_prot, agg_for_prot], dim=1))))

        if self.use_l2norm:
            h_rna_new  = F.normalize(h_rna_new,  p=2, dim=1)
            h_prot_new = F.normalize(h_prot_new, p=2, dim=1)

        return h_rna_new, h_prot_new

class ProtoGNN_GraphSAGE(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 sample_k=25, edge_drop=0.0, use_l2norm=True):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop
        self.use_l2norm = use_l2norm
        self.sample_k = sample_k

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        self.sage_layers = nn.ModuleList([
            BipartiteSAGELayer(d, dropout=dropout, sample_k=sample_k, use_l2norm=use_l2norm)
            for _ in range(layers)
        ])

        self.pair_gate = nn.Sequential(
            nn.Linear(d * 4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1)
        )
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        in_edge = (d * 4) + (d * 2)
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot=None, prot2rna=None):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        n_rna  = x_rna.size(0)
        n_prot = x_prot.size(0)
        device = x_rna.device

        if self.training and self.edge_drop > 0:
            A_dropped = sparse_edge_dropout(A_rp, self.edge_drop, training=True)
        else:
            A_dropped = A_rp

        rna2prot_use, prot2rna_use = build_neighbour_lists(A_dropped, n_rna, n_prot, device)

        h_r, h_p = h0_r, h0_p
        for layer in self.sage_layers:
            h_r, h_p = layer(h_r, h_p, rna2prot_use, prot2rna_use)

        self._cache_h0 = (h0_r, h0_p)
        dummy_core = torch.zeros(1, device=device)
        return h_r, h_p, dummy_core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]; hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids]; hp0 = h0_p[p_ids]
        a = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0 - hp0), hr0 * hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)
        feat = torch.cat([hr, hp, torch.abs(hr - hp), hr * hp, hr0, hp0], dim=1)
        return self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)


In [ ]:
def make_pairmlp(d_rna=640, d_prot=1024, d=256, dropout=0.2):
    return PairMLPGraphWrapper(d_rna, d_prot, d, dropout)

def make_deggate(d_rna=640, d_prot=1024, d=256, layers=2,
                 dropout=0.1, edge_drop=0.2):
    return ProtoGNN_DegGate(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_weighted(d_rna=640, d_prot=1024, d=256, layers=2,
                  dropout=0.1, edge_drop=0.2):
    return ProtoGNN_WeightedBipartite(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_coneighbor(d_rna=640, d_prot=1024, d=256, layers=2,
                    dropout=0.1, edge_drop=0.2):
    return ProtoGNN_CoNeighbor(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_protocontrast(d_rna=640, d_prot=1024, d=256, layers=2,
                       dropout=0.1, edge_drop=0.2, K=4, tau=0.2, contrast_w=0.2):
    return ProtoGNN_ProtoContrast(d_rna, d_prot, d, layers, dropout,
                                  edge_drop, K, tau, contrast_w)

def make_pairmlp_wide(d_rna=640, d_prot=1024, d=256, dropout=0.2):
    return PairMLP_Wide_Wrapper(d_rna, d_prot, d, dropout)

def make_pairmlp_deep(d_rna=640, d_prot=1024, d=256, dropout=0.2):
    return PairMLP_Deep_Wrapper(d_rna, d_prot, d, dropout)

def make_pairmlp_cross(d_rna=640, d_prot=1024, d=256, dropout=0.2, n_heads=4):
    return PairMLP_Cross_Wrapper(d_rna, d_prot, d, dropout, n_heads)

#

def make_graphsage(d_rna=640, d_prot=1024, d=256, layers=2,
                   dropout=0.1, sample_k=25, edge_drop=0.2):
    return ProtoGNN_GraphSAGE(d_rna, d_prot, d, layers, dropout,
                              sample_k=sample_k, edge_drop=edge_drop)

def make_graphsage(d_rna=640, d_prot=1024, d=256, layers=2,
                   dropout=0.1, sample_k=25, edge_drop=0.2, use_l2norm=True):
    return ProtoGNN_GraphSAGE(d_rna, d_prot, d, layers, dropout,
                              sample_k=sample_k, edge_drop=edge_drop,
                              use_l2norm=use_l2norm)


### 2.5 Training, evaluation, and splits

* `train_one_fold` trains a model on one outer fold and returns the
  trained model packed with the tensors needed to score arbitrary pairs.
* `get_edge_logits` scores any `(r, p)` index arrays under a trained model.
* `build_*_coldstart_splits` produce protocol-matched outer splits.
* `build_inner_val_split` creates the protocol-matched inner validation
  split from each outer training fold.
* `run_inductive_cv` is the 5-fold driver used for the within-dataset
  MCC tables; `run_inductive_cv_with_ckpts` retains per-fold weights so
  TheNovel can be scored without retraining.

In [ ]:
@torch.no_grad()
def get_edge_logits(model, x_rna_d, x_prot_d, A_rp_d, rna2prot, prot2rna,
                    r_idx_np, p_idx_np, bs=4096, device=None):
    if device is None: device = str(next(model.parameters()).device)
    model.eval()
    h_rna, h_prot, core = model.encode_nodes(x_rna_d, x_prot_d, A_rp_d, rna2prot, prot2rna)
    r_t = torch.tensor(r_idx_np, dtype=torch.long, device=device)
    p_t = torch.tensor(p_idx_np, dtype=torch.long, device=device)
    out = []
    for i in range(0, r_t.numel(), bs):
        out.append(model.score_edges(h_rna, h_prot, core,
                                     r_t[i:i+bs], p_t[i:i+bs]).detach().cpu())
    return torch.cat(out).numpy()


In [ ]:
def train_one_fold(
    model, x_rna_np, x_prot_np, r_idx, p_idx, y,
    train_idx, val_idx, test_idx,
    epochs=50, lr=1e-3, bs=4096, early_stop_patience=5,
    device=None,
):
    """
    Generic training loop — works for BOTH PairMLPGraphWrapper and ProtoGNN_ProtoContrast.
    Returns (best_val_auroc, test_logits, val_logits, pack).
    """
    if device is None: device = str(DEVICE)
    n_rna  = x_rna_np.shape[0]; n_prot = x_prot_np.shape[0]
    x_rna_d  = torch.tensor(x_rna_np, dtype=torch.float32).to(device)
    x_prot_d = torch.tensor(x_prot_np, dtype=torch.float32).to(device)

    A_rp, rna2prot, prot2rna = build_train_graph_from_split(
        r_idx, p_idx, y, train_idx, n_rna, n_prot)
    A_rp_d = A_rp.to(device)
    model   = model.to(device)

    pos_w = torch.tensor(
        [(y[train_idx]==0).sum() / max((y[train_idx]==1).sum(), 1)],
        dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt     = torch.optim.AdamW(model.parameters(), lr=lr)

    tr_r = torch.tensor(r_idx[train_idx], dtype=torch.long)
    tr_p = torch.tensor(p_idx[train_idx], dtype=torch.long)
    tr_y = torch.tensor(y[train_idx],     dtype=torch.float32)

    best_val_auroc, best_state, bad = -1.0, None, 0

    for ep in range(1, epochs + 1):
        model.train()
        h_rna, h_prot, core = model.encode_nodes(x_rna_d, x_prot_d, A_rp_d, rna2prot, prot2rna)
        opt.zero_grad()
        perm    = torch.randperm(len(tr_y))
        tr_r_sh = tr_r[perm].to(device)
        tr_p_sh = tr_p[perm].to(device)
        tr_y_sh = tr_y[perm].to(device)
        total_n = len(tr_y); total_loss = 0.0

        for i in range(0, total_n, bs):
            rr  = tr_r_sh[i:i+bs]; pp = tr_p_sh[i:i+bs]; yy = tr_y_sh[i:i+bs]
            lg  = model.score_edges(h_rna, h_prot, core, rr, pp)
            total_loss = total_loss + loss_fn(lg, yy) * (yy.numel() / total_n)

        total_loss.backward(); opt.step()

        # ── val ──
        val_logits = get_edge_logits(model, x_rna_d, x_prot_d, A_rp_d,
                                     rna2prot, prot2rna,
                                     r_idx[val_idx], p_idx[val_idx], device=device)
        val_met = compute_metrics_from_logits(y[val_idx], val_logits)
        print(f"  ep {ep:02d} | loss={float(total_loss.detach().cpu()):.4f} "
              f"| val AUROC={val_met['AUROC']:.4f} MCC={val_met['MCC']:.4f}")

        if val_met["AUROC"] > best_val_auroc:
            best_val_auroc = val_met["AUROC"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= early_stop_patience:
                print(f"  Early stop @ ep {ep}")
                break

    model.load_state_dict(best_state)

    pack = dict(x_rna_d=x_rna_d, x_prot_d=x_prot_d, A_rp_d=A_rp_d,
                rna2prot=rna2prot, prot2rna=prot2rna, device=device)

    val_logits  = get_edge_logits(model, **pack,
                                  r_idx_np=r_idx[val_idx],  p_idx_np=p_idx[val_idx])
    test_logits = get_edge_logits(model, **pack,
                                  r_idx_np=r_idx[test_idx], p_idx_np=p_idx[test_idx])
    return best_val_auroc, test_logits, val_logits, pack


In [ ]:
def build_rna_coldstart_splits(r_idx, p_idx, y, n_splits=5, seed=42):
    """
    RNA-cold-start: held-out RNAs are completely unseen during training.
    Every edge involving a test-fold RNA is in the test set.

    Returns list of (train_edge_indices, test_edge_indices).
    """
    unique_rnas = np.unique(r_idx)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for tr_rna_pos, te_rna_pos in kf.split(unique_rnas):
        train_rnas = set(unique_rnas[tr_rna_pos].tolist())
        test_rnas  = set(unique_rnas[te_rna_pos].tolist())
        train_mask = np.array([r in train_rnas for r in r_idx])
        test_mask  = np.array([r in test_rnas  for r in r_idx])
        tr = np.where(train_mask)[0]
        te = np.where(test_mask)[0]
        splits.append((tr, te))
        print(f"  [RNA-cold fold] train edges={len(tr)} "
              f"(pos={y[tr].sum()}) | test edges={len(te)} (pos={y[te].sum()}) "
              f"| test RNAs={len(test_rnas)}")
    return splits

def build_prot_coldstart_splits(r_idx, p_idx, y, n_splits=5, seed=42):
    """
    Protein-cold-start: held-out proteins are completely unseen during training.
    """
    unique_prots = np.unique(p_idx)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for tr_prot_pos, te_prot_pos in kf.split(unique_prots):
        train_prots = set(unique_prots[tr_prot_pos].tolist())
        test_prots  = set(unique_prots[te_prot_pos].tolist())
        train_mask = np.array([p in train_prots for p in p_idx])
        test_mask  = np.array([p in test_prots  for p in p_idx])
        tr = np.where(train_mask)[0]
        te = np.where(test_mask)[0]
        splits.append((tr, te))
        print(f"  [Prot-cold fold] train edges={len(tr)} "
              f"(pos={y[tr].sum()}) | test edges={len(te)} (pos={y[te].sum()}) "
              f"| test Prots={len(test_prots)}")
    return splits

def build_inner_val_split(r_idx, p_idx, y, train_idx, split_type,
                           val_frac=0.10, seed=42):
    """
    Creates an inner validation split from train_idx that MATCHES
    the outer split_type.

    edge     → random stratified edge split (original behaviour)
    rna_cold → hold out val_frac of training RNAs entirely
    prot_cold→ hold out val_frac of training proteins entirely

    Returns (inner_tr_idx, inner_va_idx) — both subsets of train_idx.
    Raises ValueError if either split ends up empty or single-class.
    """
    rng = np.random.RandomState(seed)

    if split_type == "edge":
        try:
            return train_test_split(train_idx, test_size=val_frac,
                                    random_state=seed, stratify=y[train_idx])
        except ValueError:
            return train_test_split(train_idx, test_size=val_frac,
                                    random_state=seed)

    elif split_type == "rna_cold":
        train_rnas = np.unique(r_idx[train_idx])
        n_val      = max(1, int(len(train_rnas) * val_frac))

        val_rnas   = set(rng.choice(train_rnas, size=n_val,
                                     replace=False).tolist())
        tr_rnas    = set(train_rnas.tolist()) - val_rnas

        # map back to edge-level indices (within train_idx)
        tr_mask = np.array([r_idx[i] in tr_rnas  for i in train_idx])
        va_mask = np.array([r_idx[i] in val_rnas  for i in train_idx])

        tr = train_idx[tr_mask]
        va = train_idx[va_mask]

        # sanity checks
        if len(va) == 0:
            raise ValueError("Inner RNA-cold val is empty — reduce val_frac.")
        if len(np.unique(y[va])) < 2:
            print("    Inner val single-class; falling back to edge split.")
            return train_test_split(train_idx, test_size=val_frac,
                                    random_state=seed)
        return tr, va

    elif split_type == "prot_cold":
        train_prots = np.unique(p_idx[train_idx])
        n_val       = max(1, int(len(train_prots) * val_frac))

        val_prots   = set(rng.choice(train_prots, size=n_val,
                                      replace=False).tolist())
        tr_prots    = set(train_prots.tolist()) - val_prots

        tr_mask = np.array([p_idx[i] in tr_prots  for i in train_idx])
        va_mask = np.array([p_idx[i] in val_prots  for i in train_idx])

        tr = train_idx[tr_mask]
        va = train_idx[va_mask]

        if len(va) == 0:
            raise ValueError("Inner prot-cold val is empty — reduce val_frac.")
        if len(np.unique(y[va])) < 2:
            print("    Inner val single-class; falling back to edge split.")
            return train_test_split(train_idx, test_size=val_frac,
                                    random_state=seed)
        return tr, va

    else:
        raise ValueError(f"Unknown split_type: {split_type}")


In [ ]:
def run_inductive_cv(
    model_factory,          
    dataset="NPInter2",
    split_type="rna_cold",  # "rna_cold" | "prot_cold" | "edge"
    n_splits=5,
    val_frac=0.1,
    seed=64,
    epochs=50,
    early_stop_patience=5,
    lr=1e-3,
    bs=4096,
    device=None,
):

    if device is None: device = str(DEVICE)

    x_rna, x_prot, r_idx, p_idx, y, meta = build_graph_data(dataset)

    if split_type == "rna_cold":
        splits = build_rna_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    elif split_type == "prot_cold":
        splits = build_prot_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    elif split_type == "edge":
        # standard edge-level stratified CV (for comparison)
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(skf.split(np.arange(len(y)), y))
    else:
        raise ValueError(f"Unknown split_type='{split_type}'. Use rna_cold | prot_cold | edge.")

    fold_rows = []

    for fold, (tr_full, te) in enumerate(splits):
        print(f"\n=== [{split_type}] {dataset} | fold {fold+1}/{n_splits} ===")

        # ── inner val split from train ──────────────────────────────────────
        # For cold-start: stratify only if both classes present
        tr, va = build_inner_val_split(
            r_idx, p_idx, y, tr_full,
            split_type=split_type,
            val_frac=val_frac,
            seed=seed
        )
        print(f"  inner_train={len(tr)} inner_val={len(va)} "
              f"| inner_val_pos={y[va].sum()} inner_val_neg={(y[va]==0).sum()}")

        # ── train ────────────────────────────────────────────────────────────
        model = model_factory()
        best_val_auroc, test_logits, val_logits, pack = train_one_fold(
            model, x_rna, x_prot, r_idx, p_idx, y,
            train_idx=tr, val_idx=va, test_idx=te,
            epochs=epochs, lr=lr, bs=bs,
            early_stop_patience=early_stop_patience, device=device,
        )

        # ── MCC-tuned threshold on val ───────────────────────────────────────
        thr, best_mcc_val = best_thr_by_mcc(y[va], val_logits)

        # ── test metrics at tuned threshold ──────────────────────────────────
        # Skip test folds with only one class (can happen in cold-start)
        if len(np.unique(y[te])) < 2:
            print(f"   Fold {fold}: test set has only one class — skipping.")
            continue

        test_met = compute_metrics_from_logits(y[te], test_logits, thr=thr)
        print(f"  [Fold {fold}] val_AUROC={best_val_auroc:.4f} "
              f"val_MCC={best_mcc_val:.4f}@thr={thr:.3f}")
        print(f"  [Fold {fold}] TEST "
              + " ".join(f"{k}={v:.4f}" for k, v in test_met.items()))

        fold_rows.append({"fold": fold, "thr": thr,
                          "best_val_auroc": best_val_auroc, **test_met})

    # ── summary ──────────────────────────────────────────────────────────────
    def ms(key):
        arr = np.array([r[key] for r in fold_rows], dtype=float)
        return float(np.nanmean(arr)), float(np.nanstd(arr))

    keys    = ["ACC", "SEN", "SPE", "PRE", "MCC", "AUROC", "AUPRC"]
    summary = {k: ms(k) for k in keys}

    print(f"\n=== Summary [{split_type}] {dataset} (mean ± std, {len(fold_rows)} folds) ===")
    for k in keys:
        m, s = summary[k]
        print(f"  {k}: {m:.4f} ± {s:.4f}")

    return fold_rows, summary


In [ ]:
def paired_ttest(fold_rows_a, fold_rows_b, metric="MCC", label_a="A", label_b="B"):
    """Paired t-test across fold scores. 5 folds → low power; interpret carefully."""
    a = np.array([r[metric] for r in fold_rows_a])
    b = np.array([r[metric] for r in fold_rows_b])
    t, p = scipy_stats.ttest_rel(a, b)
    print(f"\n[Paired t-test] {metric}")
    print(f"  {label_a}: {a.mean():.4f} ± {a.std():.4f}")
    print(f"  {label_b}: {b.mean():.4f} ± {b.std():.4f}")
    print(f"  t={t:.3f}  two-sided p={p:.4f}  one-sided p={p/2:.4f}")
    return t, p


In [ ]:
def run_inductive_cv_with_ckpts(
    model_factory,
    dataset="NPInter2",
    split_type="rna_cold",
    n_splits=5,
    val_frac=0.1,
    seed=64,
    epochs=50,
    early_stop_patience=5,
    lr=1e-3,
    bs=4096,
    device=None,
):
    if device is None:
        device = str(DEVICE)

    x_rna, x_prot, r_idx, p_idx, y, meta = build_graph_data(dataset)

    if split_type == "rna_cold":
        splits = build_rna_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    elif split_type == "prot_cold":
        splits = build_prot_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    elif split_type == "edge":
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = list(skf.split(np.arange(len(y)), y))
    else:
        raise ValueError(f"Unknown split_type='{split_type}'.")

    fold_rows = []
    fold_checkpoints = []

    for fold, (tr_full, te) in enumerate(splits):
        print(f"\n=== [{split_type}] {dataset} | fold {fold+1}/{n_splits} ===")

        tr, va = build_inner_val_split(
            r_idx, p_idx, y, tr_full,
            split_type=split_type,
            val_frac=val_frac,
            seed=seed,
        )
        print(f"  inner_train={len(tr)} inner_val={len(va)} "
              f"| inner_val_pos={y[va].sum()} inner_val_neg={(y[va]==0).sum()}")

        model = model_factory()
        best_val_auroc, test_logits, val_logits, pack = train_one_fold(
            model, x_rna, x_prot, r_idx, p_idx, y,
            train_idx=tr, val_idx=va, test_idx=te,
            epochs=epochs, lr=lr, bs=bs,
            early_stop_patience=early_stop_patience, device=device,
        )

        # ── CAPTURE THE BEST STATE ───────────────────────────────────
        # train_one_fold should have already loaded best_state into model
        # via early stopping. So model.state_dict() right now IS the best state.
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        thr, best_mcc_val = best_thr_by_mcc(y[va], val_logits)

        if len(np.unique(y[te])) < 2:
            print(f"   Fold {fold}: test set has only one class — skipping.")
            continue

        test_met = compute_metrics_from_logits(y[te], test_logits, thr=thr)
        print(f"  [Fold {fold}] val_AUROC={best_val_auroc:.4f} "
              f"val_MCC={best_mcc_val:.4f}@thr={thr:.3f}")
        print(f"  [Fold {fold}] TEST "
              + " ".join(f"{k}={v:.4f}" for k, v in test_met.items()))

        fold_rows.append({"fold": fold, "thr": thr,
                          "best_val_auroc": best_val_auroc, **test_met})
        fold_checkpoints.append(best_state)

    def ms(key):
        arr = np.array([r[key] for r in fold_rows], dtype=float)
        return float(np.nanmean(arr)), float(np.nanstd(arr))

    keys    = ["ACC", "SEN", "SPE", "PRE", "MCC", "AUROC", "AUPRC"]
    summary = {k: ms(k) for k in keys}

    print(f"\n=== Summary [{split_type}] {dataset} (mean ± std, {len(fold_rows)} folds) ===")
    for k in keys:
        m, s = summary[k]
        print(f"  {k}: {m:.4f} ± {s:.4f}")

    return fold_rows, summary, fold_checkpoints


### 2.6 TheNovel (cross-dataset) loaders and scorers

`load_thenovel_data` reads the NPInter5-derived hold-out benchmark
released with ZHMolGraph and re-uses its precomputed embeddings.
`score_thenovel_with_model` scores the held-out pairs using a trained
fold model; `evaluate_thenovel_ensemble` averages logits across folds.

In [ ]:
"""
TheNovel evaluation 

  1. Train models on NPInter2 with 5-fold CV (already done — checkpoints needed).
  2. For each fold's trained model, score all TheNovel edges.
  3. Average the 5 per-fold scores per edge.
  4. Compute AUROC, AUPRC, MCC against TheNovel labels.

This matches ZHMolGraph's TheNovel evaluation protocol exactly: train on NPInter2,
test on the disjoint NPInter5-derived set, ensemble across the 5 CV models.

"""

import os
import pickle as pkl
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef

# 1. CONFIG

THENOVEL_NAME = "NPInter5"  

THENOVEL_FILES = {
    "rna_pkl":  f"RPI_{THENOVEL_NAME}_rnafm_embed_normal.pkl",
    "prot_pkl": f"RPI_{THENOVEL_NAME}_proteinprottrans_embed_normal.pkl",
    "pos_csv":  f"dataset_RPI_{THENOVEL_NAME}_RP.csv",
    "neg_xlsx": f"{THENOVEL_NAME}.xlsx",   # optional
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 2. LOAD THENOVEL DATA


def _accession_prefix(name):
    """Split 'NONHSAG040596-7SK' -> 'NONHSAG040596'. Strips whitespace."""
    if name is None:
        return None
    s = str(name).strip()
    if "-" in s:
        s = s.split("-", 1)[0]
    return s

def load_thenovel_data(base_dir=None):
    """
    Joins NPInter5 CSV and xlsx by ACCESSION PREFIX (part before the first '-').
    """
    base = base_dir if base_dir is not None else BASE
    paths = {k: os.path.join(base, v) for k, v in THENOVEL_FILES.items()}

    # ── 1. Embedding pickles ─────────────────────────────────────────
    with open(paths["rna_pkl"], "rb") as f:
        rna_df = pkl.load(f)
    with open(paths["prot_pkl"], "rb") as f:
        prot_df = pkl.load(f)

    rna_seq_col_emb  = pick_col(rna_df,  ["RNA_aa_code",    "RNA_seq",     "rna_seq"])
    prot_seq_col_emb = pick_col(prot_df, ["target_aa_code", "protein_seq", "prot_seq"])
    emb_col          = pick_col(rna_df,  ["normalized_embeddings", "embeddings", "embed"])

    x_rna_novel  = np.stack(rna_df[emb_col].to_list()).astype(np.float32)
    x_prot_novel = np.stack(prot_df[emb_col].to_list()).astype(np.float32)

    rna_seqs  = rna_df[rna_seq_col_emb].astype(str).tolist()
    prot_seqs = prot_df[prot_seq_col_emb].astype(str).tolist()
    rna_map   = {s: i for i, s in enumerate(rna_seqs)}
    prot_map  = {s: i for i, s in enumerate(prot_seqs)}

    print(f"[TheNovel] {len(x_rna_novel)} RNAs, {len(x_prot_novel)} proteins in embedding files")

    # ── 2. CSV: build accession -> sequence lookup ───────────────────
    csv_df = pd.read_csv(paths["pos_csv"])
    print(f"[TheNovel] CSV rows: {len(csv_df)}")

    rna_name_col_csv  = pick_col(csv_df, ["RNA names",  "rna names",  "RNA_name", "RNA"])
    prot_name_col_csv = pick_col(csv_df, ["Protein names", "protein names", "Protein_name", "Protein"])
    rna_seq_col_csv   = pick_col(csv_df, ["RNA_aa_code", "rna_seq", "RNA_seq"])
    prot_seq_col_csv  = pick_col(csv_df, ["target_aa_code", "protein_seq", "prot_seq"])

    # CSV names are already accessions; just strip whitespace
    csv_df["_rna_key"]  = csv_df[rna_name_col_csv].apply(_accession_prefix)
    csv_df["_prot_key"] = csv_df[prot_name_col_csv].apply(_accession_prefix)

    rna_acc_to_seq = (
        csv_df[["_rna_key", rna_seq_col_csv]]
        .dropna()
        .drop_duplicates(subset=["_rna_key"])
        .set_index("_rna_key")[rna_seq_col_csv]
        .astype(str)
        .to_dict()
    )
    prot_acc_to_seq = (
        csv_df[["_prot_key", prot_seq_col_csv]]
        .dropna()
        .drop_duplicates(subset=["_prot_key"])
        .set_index("_prot_key")[prot_seq_col_csv]
        .astype(str)
        .to_dict()
    )
    print(f"[TheNovel] CSV lookup: {len(rna_acc_to_seq)} RNA accessions, "
          f"{len(prot_acc_to_seq)} protein accessions")

    # ── 3. xlsx: split compound names, take accession prefix ─────────
    xlsx_df = pd.read_excel(paths["neg_xlsx"])
    print(f"[TheNovel] xlsx rows: {len(xlsx_df)}, "
          f"label dist: pos={int((xlsx_df['Labels']==1).sum())}, "
          f"neg={int((xlsx_df['Labels']==0).sum())}")

    rna_name_col_xl  = pick_col(xlsx_df, ["RNA names",  "rna names",  "RNA_name", "RNA"])
    prot_name_col_xl = pick_col(xlsx_df, ["Protein names", "protein names", "Protein_name", "Protein"])
    label_col_xl     = pick_col(xlsx_df, ["Labels", "labels", "Label", "Y", "y"])

    # ── 4. Join via accession prefix ─────────────────────────────────
    r_idx_list, p_idx_list, y_list = [], [], []
    n_skipped_no_acc = 0
    n_skipped_no_seq = 0
    examples_skipped = []

    for _, row in xlsx_df.iterrows():
        rna_acc  = _accession_prefix(row[rna_name_col_xl])
        prot_acc = _accession_prefix(row[prot_name_col_xl])
        label    = int(row[label_col_xl])

        rna_seq  = rna_acc_to_seq.get(rna_acc)
        prot_seq = prot_acc_to_seq.get(prot_acc)
        if rna_seq is None or prot_seq is None:
            n_skipped_no_acc += 1
            if len(examples_skipped) < 3:
                missing = "rna" if rna_seq is None else "prot"
                examples_skipped.append((rna_acc, prot_acc, missing))
            continue

        r_idx = rna_map.get(rna_seq)
        p_idx = prot_map.get(prot_seq)
        if r_idx is None or p_idx is None:
            n_skipped_no_seq += 1
            continue

        r_idx_list.append(r_idx)
        p_idx_list.append(p_idx)
        y_list.append(label)

    if n_skipped_no_acc > 0:
        print(f"[TheNovel] Skipped {n_skipped_no_acc} rows: accession not in CSV")
        for rn, pn, which in examples_skipped:
            print(f"    e.g. RNA-acc={rn!r}, Prot-acc={pn!r} (missing: {which})")
    if n_skipped_no_seq > 0:
        print(f"[TheNovel] Skipped {n_skipped_no_seq} rows: sequence not in embedding map")

    r_idx_arr = np.array(r_idx_list, dtype=np.int64)
    p_idx_arr = np.array(p_idx_list, dtype=np.int64)
    y_arr     = np.array(y_list,     dtype=np.int64)

    n_pos = int((y_arr == 1).sum())
    n_neg = int((y_arr == 0).sum())
    print(f"[TheNovel] Final eval set: {len(y_arr)} edges (pos={n_pos}, neg={n_neg})")

    return x_rna_novel, x_prot_novel, r_idx_arr, p_idx_arr, y_arr

def _load_xlsx_negatives(neg_path, rna_map, prot_map):
    """Parse ZHMolGraph's NPInter5.xlsx negative file. Format:
       columns include 'RNA names', 'Protein names', 'Labels', plus seq fields."""
    df = pd.read_excel(neg_path)
    print(df.columns)

   
    rna_seq_col  = None
    prot_seq_col = None
    label_col    = None
    for c in df.columns:
        if c.lower() in ("rna_aa_code", "rna_seq", "rna names"):       rna_seq_col = c
        if c.lower() in ("target_aa_code", "protein_seq", "prot_seq", "protein names"): prot_seq_col = c
        if c.lower() in ("labels", "label", "y"):          label_col = c

    if rna_seq_col is None or prot_seq_col is None or label_col is None:
       
        print("NO")
        
        return []

    df = df[df[label_col].astype(int) == 0]
    out = []
    for _, row in df.iterrows():
        r = rna_map.get(str(row[rna_seq_col]))
        p = prot_map.get(str(row[prot_seq_col]))
        if r is not None and p is not None:
            out.append((r, p))
    return out

def _sample_random_negatives(pos_df, n_rna, n_prot, n_neg, seed=64):
    """Sample n_neg random (r, p) pairs that are NOT in pos_df."""
    rng = np.random.default_rng(seed)
    pos_set = set(zip(pos_df["r_idx"].tolist(), pos_df["p_idx"].tolist()))
    out, attempts = [], 0
    while len(out) < n_neg and attempts < n_neg * 50:
        r = int(rng.integers(0, n_rna))
        p = int(rng.integers(0, n_prot))
        if (r, p) not in pos_set:
            out.append((r, p))
        attempts += 1
    return out

# 3. SCORE TheNovel WITH ONE TRAINED MODEL

def score_thenovel_with_model(model, x_rna_novel, x_prot_novel,
                              r_idx, p_idx, batch_size=4096):
    model.eval()
    n_rna  = x_rna_novel.shape[0]
    n_prot = x_prot_novel.shape[0]

    # Build empty bipartite adjacency: (n_rna, n_prot) sparse with no edges
    indices = torch.zeros((2, 0), dtype=torch.long, device=DEVICE)
    values  = torch.zeros((0,),    dtype=torch.float32, device=DEVICE)
    A_rp = torch.sparse_coo_tensor(indices, values, (n_rna, n_prot)).coalesce().to(DEVICE)

    # Per-node neighbour lists (empty for all nodes)
    rna2prot = [torch.zeros(0, dtype=torch.long, device=DEVICE) for _ in range(n_rna)]
    prot2rna = [torch.zeros(0, dtype=torch.long, device=DEVICE) for _ in range(n_prot)]

    x_rna_t  = torch.from_numpy(x_rna_novel).to(DEVICE)
    x_prot_t = torch.from_numpy(x_prot_novel).to(DEVICE)

    with torch.no_grad():
        # Encode all nodes once
        h_rna, h_prot, core = model.encode_nodes(x_rna_t, x_prot_t, A_rp, rna2prot, prot2rna)

        # Score edges in batches
        all_logits = []
        n_edges = len(r_idx)
        for start in range(0, n_edges, batch_size):
            end = min(start + batch_size, n_edges)
            r_batch = torch.from_numpy(r_idx[start:end]).long().to(DEVICE)
            p_batch = torch.from_numpy(p_idx[start:end]).long().to(DEVICE)
            logits = model.score_edges(h_rna, h_prot, core, r_batch, p_batch)
            all_logits.append(logits.cpu().numpy())

    return np.concatenate(all_logits)

# 4. ENSEMBLE OVER 5 CV MODELS  (matches ZHMolGraph's TheNovel protocol)

def evaluate_thenovel_ensemble(model_factory, fold_checkpoints,
                               x_rna_novel, x_prot_novel,
                               r_idx, p_idx, y, label="ProtoContrast"):
    n_folds = len(fold_checkpoints)
    fold_logits = []   

    for fold, state_dict in enumerate(fold_checkpoints):
        model = model_factory().to(DEVICE)
        model.load_state_dict(state_dict)
        logits = score_thenovel_with_model(model, x_rna_novel, x_prot_novel,
                                           r_idx, p_idx)
        fold_logits.append(logits)

        # Per-fold metrics
        auroc = roc_auc_score(y, logits)
        auprc = average_precision_score(y, logits)
        print(f"  fold {fold}: AUROC={auroc:.4f}  AUPRC={auprc:.4f}")

    # Ensemble: average the logits across folds
    mean_logits = np.mean(np.stack(fold_logits, axis=0), axis=0)

    auroc = roc_auc_score(y, mean_logits)
    auprc = average_precision_score(y, mean_logits)

    thr_grid = np.linspace(mean_logits.min(), mean_logits.max(), 200)
    mcc_vals = [matthews_corrcoef(y, (mean_logits > t).astype(int)) for t in thr_grid]
    best_idx = int(np.argmax(mcc_vals))
    best_mcc = mcc_vals[best_idx]
    best_thr = thr_grid[best_idx]

    print(f"\n[{label} ensemble] AUROC={auroc:.4f}  AUPRC={auprc:.4f}  "
          f"MCC={best_mcc:.4f} (thr={best_thr:.3f})")

    return {
        "label": label,
        "AUROC": auroc,
        "AUPRC": auprc,
        "MCC":   best_mcc,
        "threshold": best_thr,
        "fold_logits": fold_logits,
        "ensemble_logits": mean_logits,
    }


## 3. Diagnostics

`inspect_prot_cold_folds` reports per-fold protein-degree statistics that
motivate the hub-and-tail discussion in the paper.

In [ ]:
def inspect_prot_cold_folds(r_idx, p_idx, y, splits, dataset_name=""):
    print(f"\n=== Prot-cold fold diagnostics: {dataset_name} ===")
    for fold, (tr, te) in enumerate(splits):
        te_prots     = np.unique(p_idx[te])
        # how many edges does each test protein have in the TEST set
        te_pos_per_prot = [(p_idx[te][y[te]==1] == pp).sum() for pp in te_prots]
        te_neg_per_prot = [(p_idx[te][y[te]==0] == pp).sum() for pp in te_prots]
        # how many edges does each test protein have in TRAIN
        # (should be 0 by construction but verify)
        tr_count_per_prot = [(p_idx[tr] == pp).sum() for pp in te_prots]

        print(f"\n  Fold {fold}:")
        print(f"    test edges: {len(te)}  pos={y[te].sum()}  neg={(y[te]==0).sum()}")
        print(f"    unique test prots: {len(te_prots)}")
        print(f"    test prot appearances in train (should all be 0): "
              f"max={max(tr_count_per_prot)}")
        print(f"    pos edges per test prot — "
              f"min={min(te_pos_per_prot)} max={max(te_pos_per_prot)} "
              f"mean={np.mean(te_pos_per_prot):.1f}")
        print(f"    prots with 0 positive test edges: "
              f"{sum(p==0 for p in te_pos_per_prot)}/{len(te_prots)}")

# Run it on the splits
x_rna, x_prot, r_idx, p_idx, y, meta = build_graph_data("NPInter2")

prot_splits = build_prot_coldstart_splits(r_idx, p_idx, y, n_splits=5, seed=64)
inspect_prot_cold_folds(r_idx, p_idx, y, prot_splits, dataset_name="NPInter2")


The next cell prints the protein-degree statistics quoted in the
paper's Methods section (max, mean, median, singleton count, top-10 hubs).
It assumes `y`, `p_idx` and `Counter` are in scope from the most recent
`build_graph_data` call above.

In [ ]:
# Check degree distribution of all 439 proteins in NPInter2

all_pos_mask = (y == 1)
prot_degrees = Counter(p_idx[all_pos_mask].tolist())

degrees = np.array(list(prot_degrees.values()))
print(f"Total proteins with known interactions: {len(degrees)}")
print(f"Degree distribution:")
print(f"  min={degrees.min()}  max={degrees.max()}  "
      f"mean={degrees.mean():.1f}  median={np.median(degrees):.0f}")
print(f"  Proteins with degree >= 100: {(degrees >= 100).sum()}")
print(f"  Proteins with degree >= 500: {(degrees >= 500).sum()}")
print(f"  Proteins with degree == 1:   {(degrees == 1).sum()}")
print(f"\nTop 10 hub proteins by degree:")
for pid, deg in sorted(prot_degrees.items(), key=lambda x: -x[1])[:10]:
    print(f"  protein_idx={pid}  degree={deg}")


## 4. Within-dataset ablation → Tables I, II, III

For each dataset (NPInter2, RPI7317) we run all nine models under three
protocols (edge, RNA-cold, protein-cold) with 5-fold CV. Fold-level MCC
values feed the paired t-tests in Table III via `paired_ttest`.

### 4.1 NPInter2

In [ ]:
DATASET = "NPInter2"   # change to "RPI7317" for second dataset

# Factory functions for all 5 model variants

def make_pairmlp(d_rna=640, d_prot=1024, d=256, dropout=0.2):
    """No graph, no prototypes. Pure embedding baseline."""
    return PairMLPGraphWrapper(d_rna, d_prot, d, dropout)

def make_deggate(d_rna=640, d_prot=1024, d=256, layers=2,
                 dropout=0.1, edge_drop=0.2):
    """GNN + degree gate + pair gate. No prototypes, no contrastive term.
    Uses TypeAwareCoreFusion (single global core vector)."""
    return ProtoGNN_DegGate(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_weighted(d_rna=640, d_prot=1024, d=256, layers=2,
                  dropout=0.1, edge_drop=0.2):
    """Attention-weighted bipartite message passing.
    No degree gate, no prototypes. Uses TypeAwareCoreFusion."""
    return ProtoGNN_WeightedBipartite(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_coneighbor(d_rna=640, d_prot=1024, d=256, layers=2,
                    dropout=0.1, edge_drop=0.2):
    """1-hop + 2-hop co-neighbor propagation.
    No degree gate, no prototypes. Uses TypeAwareCoreFusion."""
    return ProtoGNN_CoNeighbor(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_protocontrast(d_rna=640, d_prot=1024, d=256, layers=2,
                       dropout=0.1, edge_drop=0.2, K=4, tau=0.2, contrast_w=0.2):
    """Full model: GNN + degree gate + pair gate + prototypes + contrastive logit.
    Most expressive transductively, most brittle inductively."""
    return ProtoGNN_ProtoContrast(d_rna, d_prot, d, layers, dropout,
                                  edge_drop, K, tau, contrast_w)


MODELS = [
    ("PairMLP",            make_pairmlp),
    ("PairMLP-Wide",    make_pairmlp_wide),
    ("PairMLP-Deep",    make_pairmlp_deep),
    ("PairMLP-Cross",   make_pairmlp_cross),
    ("DegGate",            make_deggate),
    ("WeightedBipartite",  make_weighted),
    ("CoNeighbor",         make_coneighbor),
    ("ProtoContrast",      make_protocontrast),
]

SPLIT_TYPES = ["edge", "rna_cold", "prot_cold"]

import numpy as np

all_results = {}   

for model_name, factory in MODELS:
    for split in SPLIT_TYPES:
        tag = f"{model_name} | {split}"
        print("\n" + "="*60)
        print(f"{tag} | {DATASET}")
        print("="*60)

        rows, summary = run_inductive_cv(
            model_factory=factory,
            dataset=DATASET,
            split_type=split,
            n_splits=5, seed=64, epochs=50, early_stop_patience=5,
            lr=1e-3, bs=4096,
        )
        all_results[(model_name, split)] = {"rows": rows, "summary": summary}

# Summary table
print("\n" + "="*80)
print(f"FULL ABLATION SUMMARY — {DATASET}")
print("="*80)
print(f"{'Model':<22} {'Edge MCC':>10} {'RNA-cold MCC':>14} {'Prot-cold MCC':>15}")
print("-"*80)

for model_name, _ in MODELS:
    e_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "edge")]["rows"]])
    r_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "rna_cold")]["rows"]])
    p_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "prot_cold")]["rows"]])
    print(f"{model_name:<22} {e_mcc:>10.4f} {r_mcc:>14.4f} {p_mcc:>15.4f}")

# Pairwise significance tests: each GNN variant vs PairMLP
print("\n" + "="*80)
print("SIGNIFICANCE TESTS vs PairMLP")
print("="*80)

for model_name, _ in MODELS:
    if model_name == "PairMLP":
        continue
    for split in SPLIT_TYPES:
        print(f"\n--- {model_name} vs PairMLP | {split} ---")
        paired_ttest(
            all_results[("PairMLP", split)]["rows"],
            all_results[(model_name, split)]["rows"],
            metric="MCC", label_a="PairMLP", label_b=model_name,
        )

# Also test adjacent pairs in the ablation ladder
print("\n" + "="*80)
print("ABLATION LADDER — adjacent-pair significance (edge-level)")
print("="*80)
ladder = ["PairMLP", "DegGate", "ProtoContrast"]
for i in range(len(ladder)-1):
    a, b = ladder[i], ladder[i+1]
    print(f"\n--- {a} → {b} | edge ---")
    paired_ttest(
        all_results[(a, "edge")]["rows"],
        all_results[(b, "edge")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )
    print(f"--- {a} → {b} | rna_cold ---")
    paired_ttest(
        all_results[(a, "rna_cold")]["rows"],
        all_results[(b, "rna_cold")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )
    print(f"--- {a} → {b} | prot_cold ---")
    paired_ttest(
        all_results[(a, "prot_cold")]["rows"],
        all_results[(b, "prot_cold")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )


### 4.2 RPI7317

In [ ]:
DATASET = "RPI7317"   # change to "RPI7317" for second dataset


def make_pairmlp(d_rna=640, d_prot=1024, d=256, dropout=0.2):
    """No graph, no prototypes. Pure embedding baseline."""
    return PairMLPGraphWrapper(d_rna, d_prot, d, dropout)

def make_deggate(d_rna=640, d_prot=1024, d=256, layers=2,
                 dropout=0.1, edge_drop=0.2):
    """GNN + degree gate + pair gate. No prototypes, no contrastive term.
    Uses TypeAwareCoreFusion (single global core vector)."""
    return ProtoGNN_DegGate(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_weighted(d_rna=640, d_prot=1024, d=256, layers=2,
                  dropout=0.1, edge_drop=0.2):
    """Attention-weighted bipartite message passing.
    No degree gate, no prototypes. Uses TypeAwareCoreFusion."""
    return ProtoGNN_WeightedBipartite(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_coneighbor(d_rna=640, d_prot=1024, d=256, layers=2,
                    dropout=0.1, edge_drop=0.2):
    """1-hop + 2-hop co-neighbor propagation.
    No degree gate, no prototypes. Uses TypeAwareCoreFusion."""
    return ProtoGNN_CoNeighbor(d_rna, d_prot, d, layers, dropout, edge_drop=edge_drop)

def make_protocontrast(d_rna=640, d_prot=1024, d=256, layers=2,
                       dropout=0.1, edge_drop=0.2, K=4, tau=0.2, contrast_w=0.2):
    """Full model: GNN + degree gate + pair gate + prototypes + contrastive logit.
    Most expressive transductively, most brittle inductively."""
    return ProtoGNN_ProtoContrast(d_rna, d_prot, d, layers, dropout,
                                  edge_drop, K, tau, contrast_w)


MODELS = [
    ("PairMLP",            make_pairmlp),
    ("PairMLP-Wide",    make_pairmlp_wide),
    ("PairMLP-Deep",    make_pairmlp_deep),
    ("PairMLP-Cross",   make_pairmlp_cross),
    ("DegGate",            make_deggate),
    ("WeightedBipartite",  make_weighted),
    ("CoNeighbor",         make_coneighbor),
    ("ProtoContrast",      make_protocontrast),
]

SPLIT_TYPES = ["edge", "rna_cold", "prot_cold"]

import numpy as np


all_results = {}   

for model_name, factory in MODELS:
    for split in SPLIT_TYPES:
        tag = f"{model_name} | {split}"
        print("\n" + "="*60)
        print(f"{tag} | {DATASET}")
        print("="*60)

        rows, summary = run_inductive_cv(
            model_factory=factory,
            dataset=DATASET,
            split_type=split,
            n_splits=5, seed=64, epochs=50, early_stop_patience=5,
            lr=1e-3, bs=4096,
        )
        all_results[(model_name, split)] = {"rows": rows, "summary": summary}

# Summary table
print("\n" + "="*80)
print(f"FULL ABLATION SUMMARY — {DATASET}")
print("="*80)
print(f"{'Model':<22} {'Edge MCC':>10} {'RNA-cold MCC':>14} {'Prot-cold MCC':>15}")
print("-"*80)

for model_name, _ in MODELS:
    e_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "edge")]["rows"]])
    r_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "rna_cold")]["rows"]])
    p_mcc = np.mean([r["MCC"] for r in all_results[(model_name, "prot_cold")]["rows"]])
    print(f"{model_name:<22} {e_mcc:>10.4f} {r_mcc:>14.4f} {p_mcc:>15.4f}")

# Pairwise significance tests: each GNN variant vs PairMLP
print("\n" + "="*80)
print("SIGNIFICANCE TESTS vs PairMLP")
print("="*80)

for model_name, _ in MODELS:
    if model_name == "PairMLP":
        continue
    for split in SPLIT_TYPES:
        print(f"\n--- {model_name} vs PairMLP | {split} ---")
        paired_ttest(
            all_results[("PairMLP", split)]["rows"],
            all_results[(model_name, split)]["rows"],
            metric="MCC", label_a="PairMLP", label_b=model_name,
        )

# Also test adjacent pairs in the ablation ladder
print("\n" + "="*80)
print("ABLATION LADDER — adjacent-pair significance (edge-level)")
print("="*80)
ladder = ["PairMLP", "DegGate", "ProtoContrast"]
for i in range(len(ladder)-1):
    a, b = ladder[i], ladder[i+1]
    print(f"\n--- {a} → {b} | edge ---")
    paired_ttest(
        all_results[(a, "edge")]["rows"],
        all_results[(b, "edge")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )
    print(f"--- {a} → {b} | rna_cold ---")
    paired_ttest(
        all_results[(a, "rna_cold")]["rows"],
        all_results[(b, "rna_cold")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )
    print(f"--- {a} → {b} | prot_cold ---")
    paired_ttest(
        all_results[(a, "prot_cold")]["rows"],
        all_results[(b, "prot_cold")]["rows"],
        metric="MCC", label_a=a, label_b=b,
    )


## 5. Cross-dataset evaluation on TheNovel → Table V

Train each model on NPInter2 with 5-fold CV that retains
per-fold checkpoints, then score TheNovel using ensemble-mean logits
across the five folds.

### 5.1 Models trained on NPInter2, evaluated on TheNovel

In [ ]:
# STEP 1: Train all 5 models with 5-fold CV on NPInter2 (edge-level split)

print("\n" + "=" * 70)
print("STEP 1: Training all 5 models with 5-fold edge-level CV on NPInter2")
print("=" * 70)

all_ckpts = {}    
all_cv_rows = {}  

for model_name, factory in MODELS:
    print(f"\n{'#' * 70}")
    print(f"# Training {model_name}")
    print(f"{'#' * 70}")
    rows, summary, ckpts = run_inductive_cv_with_ckpts(
        model_factory=factory,
        dataset="NPInter2",
        split_type="edge",
        n_splits=5, seed=64, epochs=50, early_stop_patience=5,
        lr=1e-3, bs=4096,
    )
    all_ckpts[model_name] = ckpts
    all_cv_rows[model_name] = rows

# STEP 2: Load TheNovel

print("\n" + "=" * 70)
print("STEP 2: Loading TheNovel (NPInter5) data")
print("=" * 70)


x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n = load_thenovel_data(base_dir=BASE)

# STEP 3: Ensemble evaluation on TheNovel for each of the 5 models

print("\n" + "=" * 70)
print("STEP 3: TheNovel ensemble evaluation")
print("=" * 70)

thenovel_results = {}

for model_name, factory in MODELS:
    print(f"\n--- TheNovel | {model_name} ---")
    result = evaluate_thenovel_ensemble(
        factory, all_ckpts[model_name],
        x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n,
        label=model_name,
    )
    thenovel_results[model_name] = result

# STEP 4: Comparison table (all 5 models on TheNovel)

print("\n" + "=" * 70)
print("STEP 4: TheNovel comparison summary")
print("=" * 70)

print(f"\n{'Model':<22} {'AUROC':>8} {'AUPRC':>8} {'MCC':>8}")
print("-" * 50)

# ZHMolGraph reported AUROC ≈ 0.798 on TheNovel
print(f"{'ZHMolGraph (reported)':<22} {'0.798':>8} {'0.82':>8} {'~':>8}")
print("-" * 50)

for model_name, _ in MODELS:
    r = thenovel_results[model_name]
    print(f"{model_name:<22} {r['AUROC']:>8.4f} {r['AUPRC']:>8.4f} {r['MCC']:>8.4f}")


## 6. GraphSAGE tuning sweep on TheNovel → Table VI

Sweep removes per-layer L2 normalization, switches to all-neighbour
aggregation during training, combines both, and reduces depth to a
single propagation layer. Only the single-layer variant improves on the
two-layer baseline. None reaches PairMLP-Cross or ZHMolGraph.

In [ ]:
def _count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

if __name__ == "__main__":
    m = make_graphsage()
    print(f"ProtoGNN_GraphSAGE parameters: {_count_params(m):,}")


In [ ]:
# Compare GraphSAGE against the strongest baselines from prior experiments
GRAPHSAGE_COMPARISON = [
    ("PairMLP",         make_pairmlp),         # base reference
    ("PairMLP-Cross",   make_pairmlp_cross),   # current best
    ("ProtoContrast",   make_protocontrast),   # full LightGCN-based GNN
    ("GraphSAGE",       make_graphsage),       # NEW
]

SPLIT_TYPES = ["edge", "rna_cold", "prot_cold"]

graphsage_results_npinter2 = {}

print(f"\\n{'#'*70}\\n# GraphSAGE comparison on NPInter2\\n{'#'*70}")
for model_name, factory in GRAPHSAGE_COMPARISON:
    for split in SPLIT_TYPES:
        print(f"\\n{'='*70}\\n{model_name} | {split} | NPInter2\\n{'='*70}")
        rows, summary, ckpts = run_inductive_cv_with_ckpts(
            model_factory=factory,
            dataset="NPInter2",
            split_type=split,
            n_splits=5, seed=64, epochs=50, early_stop_patience=5,
            lr=1e-3, bs=4096,
        )
        graphsage_results_npinter2[(model_name, split)] = {
            "rows": rows, "summary": summary, "ckpts": ckpts,
        }

# Summary table
print(f"\\n{'='*80}\\nGraphSAGE comparison summary - NPInter2\\n{'='*80}")
print(f"{'Model':<20} {'Edge MCC':>14} {'RNA-cold':>14} {'Prot-cold':>14}")
print("-" * 80)
for name, _ in GRAPHSAGE_COMPARISON:
    e  = np.mean([r["MCC"] for r in graphsage_results_npinter2[(name, "edge")]["rows"]])
    rc = np.mean([r["MCC"] for r in graphsage_results_npinter2[(name, "rna_cold")]["rows"]])
    pc = np.mean([r["MCC"] for r in graphsage_results_npinter2[(name, "prot_cold")]["rows"]])
    print(f"{name:<20} {e:>14.4f} {rc:>14.4f} {pc:>14.4f}")

# Significance: GraphSAGE vs each other model
print(f"\\n{'='*80}\\nGraphSAGE significance tests\\n{'='*80}")
for name, _ in GRAPHSAGE_COMPARISON:
    if name == "GraphSAGE":
        continue
    for split in SPLIT_TYPES:
        print(f"\\n--- GraphSAGE vs {name} | {split} ---")
        paired_ttest(
            graphsage_results_npinter2[(name, split)]["rows"],
            graphsage_results_npinter2[("GraphSAGE", split)]["rows"],
            metric="MCC", label_a=name, label_b="GraphSAGE",
        )


In [ ]:

print(f"\\n{'='*70}\\nTheNovel evaluation — GraphSAGE vs other models\\n{'='*70}")

x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n = load_thenovel_data(base_dir=BASE)

graphsage_thenovel_results = {}

for model_name, factory in GRAPHSAGE_COMPARISON:
    print(f"\\n--- TheNovel | {model_name} ---")
    ckpts = graphsage_results_npinter2[(model_name, "edge")]["ckpts"]
    result = evaluate_thenovel_ensemble(
        factory, ckpts,
        x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n,
        label=model_name,
    )
    graphsage_thenovel_results[model_name] = result

print(f"\\n{'='*70}\\nTheNovel comparison\\n{'='*70}")
print(f"\\n{'Model':<22} {'AUROC':>8} {'AUPRC':>8} {'MCC':>8}")
print("-" * 50)
print(f"{'ZHMolGraph (reported)':<22} {'0.798':>8} {'0.820':>8} {'~':>8}")
print("-" * 50)
for name, _ in GRAPHSAGE_COMPARISON:
    r = graphsage_thenovel_results[name]
    print(f"{name:<22} {r['AUROC']:>8.4f} {r['AUPRC']:>8.4f} {r['MCC']:>8.4f}")


In [ ]:

GRAPHSAGE_TUNE_CONFIGS = [
    ("H1_NoL2",         dict(use_l2norm=False)),                       # H1
    ("H2_AllNbr",       dict(sample_k=None)),                          # H2
    ("H3_NoL2_AllNbr",  dict(sample_k=None, use_l2norm=False)),        # H3
    ("H4_OneLayer",     dict(layers=1)),                               # H4
]

# Reference baseline (already have results — no need to re-run)
BASELINE_LABEL = "Baseline_k25_L2_l2"
BASELINE_THENOVEL_AUROC = 0.7407
BASELINE_EDGE_MCC      = 0.8494

graphsage_tune_results = {}      
graphsage_tune_thenovel = {}     

print(f"\n{'#'*70}")
print(f"# GraphSAGE hyperparameter sweep — 4 configs × edge-level CV + TheNovel")
print(f"{'#'*70}")
print(f"# Reference baseline (sample_k=25, layers=2, L2-norm=True):")
print(f"#   NPInter2 Edge MCC = {BASELINE_EDGE_MCC}")
print(f"#   TheNovel AUROC    = {BASELINE_THENOVEL_AUROC}")
print(f"#   TARGET: any config gets TheNovel AUROC > 0.78")
print(f"{'#'*70}")

# Load TheNovel data once
print("\n[Sweep] Loading TheNovel data...")
x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n = load_thenovel_data(base_dir=BASE)

for label, config_kwargs in GRAPHSAGE_TUNE_CONFIGS:
    print(f"\n{'='*70}")
    print(f"CONFIG: {label}  (overrides: {config_kwargs})")
    print(f"{'='*70}")

    
    factory = partial(make_graphsage, **config_kwargs)

 
    rows, summary, ckpts = run_inductive_cv_with_ckpts(
        model_factory=factory,
        dataset="NPInter2",
        split_type="edge",
        n_splits=5, seed=64, epochs=50, early_stop_patience=5,
        lr=1e-3, bs=4096,
    )
    graphsage_tune_results[label] = {
        "rows": rows, "summary": summary, "ckpts": ckpts, "config": config_kwargs,
    }

    # TheNovel ensemble eval using the 5 CV checkpoints
    print(f"\n--- TheNovel eval | {label} ---")
    novel_result = evaluate_thenovel_ensemble(
        factory, ckpts,
        x_rna_novel, x_prot_novel, r_idx_n, p_idx_n, y_n,
        label=label,
    )
    graphsage_tune_thenovel[label] = novel_result

# SUMMARY: sorted by TheNovel AUROC

print(f"\n{'='*80}")
print(f"GraphSAGE tuning summary — sorted by TheNovel AUROC")
print(f"{'='*80}")
print(f"{'Config':<22} {'NPInter2 Edge':>14} {'TheNovel AUROC':>16} {'AUPRC':>10}")
print("-" * 80)

# Always print baseline first
print(f"{BASELINE_LABEL:<22} {BASELINE_EDGE_MCC:>14.4f} {BASELINE_THENOVEL_AUROC:>16.4f} {'~0.72':>10}")
print("-" * 80)

# Sort tuned configs by TheNovel AUROC descending
sorted_configs = sorted(
    GRAPHSAGE_TUNE_CONFIGS,
    key=lambda x: graphsage_tune_thenovel[x[0]]["AUROC"],
    reverse=True,
)
for label, _ in sorted_configs:
    edge_mcc = np.mean([r["MCC"] for r in graphsage_tune_results[label]["rows"]])
    novel    = graphsage_tune_thenovel[label]
    marker = "  <-- BEST" if label == sorted_configs[0][0] else ""
    print(f"{label:<22} {edge_mcc:>14.4f} {novel['AUROC']:>16.4f} {novel['AUPRC']:>10.4f}{marker}")

# Reference for context
print("\n" + "-" * 80)
print("Reference points:")
print(f"  ZHMolGraph (reported)  TheNovel AUROC = 0.798")
print(f"  PairMLP-Cross (ours)   TheNovel AUROC = 0.8378")
print(f"  PairMLP (base, ours)   TheNovel AUROC = 0.7935")


In [ ]:

WINNING_LABEL  = "H4_OneLayer"   
WINNING_CONFIG = dict(sample_k=None, use_l2norm=False)   

# Only run this cell if the winner's TheNovel AUROC actually beat the baseline:
winner_auroc = graphsage_tune_thenovel[WINNING_LABEL]["AUROC"]
if winner_auroc <= BASELINE_THENOVEL_AUROC:
    print(f"\n*** Winner '{WINNING_LABEL}' did NOT improve over baseline ***")
    print(f"    Best TheNovel AUROC: {winner_auroc:.4f} vs baseline {BASELINE_THENOVEL_AUROC}")
    print(f"    Skipping full evaluation. The architectural diagnosis was wrong.")
    print(f"    Conclusion: GraphSAGE-style propagation does not help RPI cold-start either.")
else:
    print(f"\n*** Winner '{WINNING_LABEL}' improved baseline ***")
    print(f"    TheNovel AUROC: {winner_auroc:.4f} vs baseline {BASELINE_THENOVEL_AUROC}")
    print(f"    Running full evaluation on all splits + RPI7317...")

    factory_winner = partial(make_graphsage, **WINNING_CONFIG)
    full_eval_results = {}

    # Add RNA-cold and prot-cold on NPInter2
    for split in ["rna_cold", "prot_cold"]:
        print(f"\n{'='*70}\n{WINNING_LABEL} | {split} | NPInter2\n{'='*70}")
        rows, summary, ckpts = run_inductive_cv_with_ckpts(
            model_factory=factory_winner,
            dataset="NPInter2",
            split_type=split,
            n_splits=5, seed=64, epochs=50, early_stop_patience=5,
            lr=1e-3, bs=4096,
        )
        full_eval_results[("NPInter2", split)] = rows

    # Reuse edge-level rows from the sweep
    full_eval_results[("NPInter2", "edge")] = graphsage_tune_results[WINNING_LABEL]["rows"]

    # All three splits on RPI7317
    for split in ["edge", "rna_cold", "prot_cold"]:
        print(f"\n{'='*70}\n{WINNING_LABEL} | {split} | RPI7317\n{'='*70}")
        rows, summary, ckpts = run_inductive_cv_with_ckpts(
            model_factory=factory_winner,
            dataset="RPI7317",
            split_type=split,
            n_splits=5, seed=64, epochs=50, early_stop_patience=5,
            lr=1e-3, bs=4096,
        )
        full_eval_results[("RPI7317", split)] = rows

    # Summary table — combine into a 2x3 ablation grid
    print(f"\n{'='*80}")
    print(f"Full evaluation: {WINNING_LABEL} on both datasets, all 3 splits")
    print(f"{'='*80}")
    print(f"{'Dataset':<12} {'Edge MCC':>14} {'RNA-cold MCC':>16} {'Prot-cold MCC':>16}")
    print("-" * 80)
    for ds in ["NPInter2", "RPI7317"]:
        e  = np.mean([r["MCC"] for r in full_eval_results[(ds, "edge")]])
        rc = np.mean([r["MCC"] for r in full_eval_results[(ds, "rna_cold")]])
        pc = np.mean([r["MCC"] for r in full_eval_results[(ds, "prot_cold")]])
        print(f"{ds:<12} {e:>14.4f} {rc:>16.4f} {pc:>16.4f}")

    # Significance vs PairMLP-Cross (the current best)
    print(f"\n{'='*80}")
    print(f"Significance: {WINNING_LABEL} vs PairMLP-Cross")
    print(f"{'='*80}")

    # Assumes we have variant_results_npinter2 from the earlier PairMLP variants notebook
    for split in ["edge", "rna_cold", "prot_cold"]:
        try:
            cross_rows = variant_results_npinter2[("PairMLP-Cross", split)]["rows"]
            print(f"\n--- {WINNING_LABEL} vs PairMLP-Cross | {split} | NPInter2 ---")
            paired_ttest(
                cross_rows,
                full_eval_results[("NPInter2", split)],
                metric="MCC", label_a="PairMLP-Cross", label_b=WINNING_LABEL,
            )
        except (NameError, KeyError):
            print(f"  (skipping — variant_results_npinter2 not in scope)")
            break


## 7. Case studies → Figures 1, 2, 3

The two case studies share a multi-seed (`[64, 1, 7]`) × multi-dataset
(`NPInter2`, `RPI7317`) driver. Per-combo intermediates are written to
`/kaggle/working/` so the runs can be resumed after a session timeout.

* **Case Study B** (protein-cold): for each held-out protein, capture
  per-test-edge logits with the held-out protein's degree and the RNA's
  training-graph degree. Produces Figures 2 and 3.
* **Case Study A** (RNA-cold retrieval): for each held-out RNA, rank
  all proteins; compute MRR and Recall@K. Produces Figure 1.

### 7.1 Configuration and helpers

In [ ]:

import os, glob, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

SEEDS     = [64, 1, 7]                       
DATASETS  = ["NPInter2", "RPI7317"]
N_SPLITS  = 5
EPOCHS    = 50
OUTDIR    = "/kaggle/working"

CS_MODELS = {
    "PairMLP-Cross": make_pairmlp_cross,
    "ProtoContrast": make_protocontrast,
    "GraphSage-2L":     make_graphsage,          # standard 2-layer 
}

# degree buckets for the held-out protein (Analysis 1)
DEG_BINS   = [0, 2, 10, 100, 10**9]
DEG_LABELS = ["1", "2-9", "10-99", "100+"]
def deg_bucket(d):
    return DEG_LABELS[np.digitize(d, DEG_BINS[1:-1])]

# RNA train-degree buckets (Analysis 2, the orphan axis)
ORPHAN_ORDER = ["orphan (deg 0)", "low (1-2)", "warm (3+)"]
def orphan_bucket(d):
    return "orphan (deg 0)" if d == 0 else ("low (1-2)" if d <= 2 else "warm (3+)")

MODEL_COLORS = {"PairMLP-Cross": "#1f77b4", "ProtoContrast": "#ff7f0e", "GraphSage-2L": "#2ca02c"}

def _agg_cell(agg, model, bucket_col, bucket, val="mean", err="std"):
    """Safely fetch (mean, std) from an aggregated table."""
    row = agg[(agg["model"] == model) & (agg[bucket_col] == bucket)]
    if len(row) == 0:
        return np.nan, 0.0
    return float(row[val].iloc[0]), float(row[err].iloc[0] if not np.isnan(row[err].iloc[0]) else 0.0)


### 7.2 Case Study B — protein-cold degree-stratified / cascade-orphan analysis

Produces Figure 2 (orphan-RNA failure) and Figure 3 (per-protein AUROC
vs held-out protein degree).

In [ ]:
# ---------------------------------------------------------------------
# CASE STUDY B: protein-cold degree-stratified & cascade-orphan analysis
# ---------------------------------------------------------------------
def capture_protcold_edges(model_factory, dataset, seed, n_splits=N_SPLITS):
    """One (model, dataset, seed): per-test-edge logits + degrees."""
    seed_all(seed)
    x_rna, x_prot, r_idx, p_idx, y, meta = build_graph_data(dataset)
    n_rna, n_prot = x_rna.shape[0], x_prot.shape[0]
    prot_full_deg = np.bincount(p_idx[y == 1], minlength=n_prot)   # hub-ness

    splits = build_prot_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    rows = []
    for fold, (tr_full, te) in enumerate(splits):
        tr, va = build_inner_val_split(r_idx, p_idx, y, tr_full,
                                       split_type="prot_cold", val_frac=0.10, seed=seed)
        rna_train_deg = np.bincount(r_idx[tr[y[tr] == 1]], minlength=n_rna)
        model = model_factory()
        _, test_logits, _, pack = train_one_fold(
            model, x_rna, x_prot, r_idx, p_idx, y,
            train_idx=tr, val_idx=va, test_idx=te,
            epochs=EPOCHS, lr=1e-3, bs=4096, early_stop_patience=5)
        for i, e in enumerate(te):
            r, p = int(r_idx[e]), int(p_idx[e])
            rows.append(dict(fold=fold, r=r, p=p, y=int(y[e]),
                             logit=float(test_logits[i]),
                             prot_deg=int(prot_full_deg[p]),
                             rna_train_deg=int(rna_train_deg[r])))
        del model, pack
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)

def run_caseB():
    t0 = time.time()
    for dataset in DATASETS:
        for name, fac in CS_MODELS.items():
            for seed in SEEDS:
                fp = f"{OUTDIR}/B_raw_{dataset}_{name}_{seed}.csv"
                if os.path.exists(fp):
                    print(f"[skip] B {dataset:9s} {name:14s} seed={seed}")
                    continue
                print(f"[run ] B {dataset:9s} {name:14s} seed={seed}")
                df = capture_protcold_edges(fac, dataset, seed)
                df["dataset"], df["model"], df["seed"] = dataset, name, seed
                df.to_csv(fp, index=False)
    raw = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUTDIR}/B_raw_*.csv")],
                    ignore_index=True)
    raw.to_csv(f"{OUTDIR}/caseB_raw_alledges.csv", index=False)
    print(f"Case B done in {(time.time()-t0)/60:.1f} min | rows={len(raw)}")
    return raw

cs_raw = run_caseB()

# ---------- Analysis 1: per-protein AUROC vs held-out protein degree ----------
recs = []
for (ds, mdl, sd, p), g in cs_raw.groupby(["dataset", "model", "seed", "p"]):
    if g["y"].nunique() < 2:
        continue
    recs.append(dict(dataset=ds, model=mdl, seed=sd, p=p,
                     deg=int(g["prot_deg"].iloc[0]),
                     auroc=roc_auc_score(g["y"], g["logit"])))
pa = pd.DataFrame(recs)
pa["bucket"] = pa["deg"].apply(deg_bucket)
pa.to_csv(f"{OUTDIR}/caseB_perprotein_auroc_raw.csv", index=False)

# per-seed bucket means -> aggregate over seeds
seed_bucket = (pa.groupby(["dataset", "model", "seed", "bucket"])["auroc"]
                 .mean().reset_index())
B1 = (seed_bucket.groupby(["dataset", "model", "bucket"])["auroc"]
        .agg(["mean", "std", "count"]).reset_index())
B1.to_csv(f"{OUTDIR}/caseB1_perprotein_summary.csv", index=False)
print("\n=== B1: per-protein AUROC vs degree (mean over seeds) ===")
print(B1.pivot_table(index=["dataset", "bucket"], columns="model", values="mean").round(3))

for dataset in DATASETS:
    fig, ax = plt.subplots(figsize=(6, 4))
    for mdl in CS_MODELS:
        sub = B1[(B1.dataset == dataset) & (B1.model == mdl)].set_index("bucket").reindex(DEG_LABELS)
        ax.errorbar(DEG_LABELS, sub["mean"], yerr=sub["std"].fillna(0),
                    marker="o", capsize=3, label=mdl, color=MODEL_COLORS[mdl])
    ax.set_xlabel("Held-out protein degree (full graph)")
    ax.set_ylabel("Per-protein AUROC")
    ax.set_title(f"Protein-cold: per-protein AUROC vs degree ({dataset})")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
    fig.savefig(f"{OUTDIR}/figB1_perprotein_auroc_{dataset}.png", dpi=200)
    plt.show()

# ---------- Analysis 2: pooled AUROC by RNA train-degree bucket (orphan axis) ----------
cs_raw["rna_bucket"] = cs_raw["rna_train_deg"].apply(orphan_bucket)
recs = []
for (ds, mdl, sd, b), g in cs_raw.groupby(["dataset", "model", "seed", "rna_bucket"]):
    au = roc_auc_score(g["y"], g["logit"]) if g["y"].nunique() > 1 else np.nan
    recs.append(dict(dataset=ds, model=mdl, seed=sd, rna_bucket=b,
                     n_edges=len(g), auroc=au))
orphan_seed = pd.DataFrame(recs)
B2 = (orphan_seed.groupby(["dataset", "model", "rna_bucket"])["auroc"]
        .agg(["mean", "std", "count"]).reset_index())
B2.to_csv(f"{OUTDIR}/caseB2_orphan_summary.csv", index=False)
print("\n=== B2: AUROC by RNA train-degree bucket (mean over seeds) ===")
print(B2.pivot_table(index=["dataset", "rna_bucket"], columns="model", values="mean").round(3))

for dataset in DATASETS:
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(ORPHAN_ORDER)); w = 0.25
    for i, mdl in enumerate(CS_MODELS):
        means = [_agg_cell(B2[B2.dataset == dataset], mdl, "rna_bucket", b)[0] for b in ORPHAN_ORDER]
        errs  = [_agg_cell(B2[B2.dataset == dataset], mdl, "rna_bucket", b)[1] for b in ORPHAN_ORDER]
        ax.bar(x + (i - 1) * w, means, w, yerr=errs, capsize=3,
               label=mdl, color=MODEL_COLORS[mdl])
    ax.axhline(0.5, ls="--", c="grey", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(ORPHAN_ORDER)
    ax.set_ylabel("Pooled AUROC"); ax.set_ylim(0.3, 1.0)
    ax.set_xlabel("RNA degree in training graph")
    ax.set_title(f"Protein-cold: orphaned-RNA failure ({dataset})")
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUTDIR}/figB2_orphan_{dataset}.png", dpi=200)
    plt.show()


### 7.3 Case Study A — RNA-cold candidate retrieval

Produces Figure 1 (MRR / Recall@K).

In [ ]:
# ---------------------------------------------------------------------
# CASE STUDY A: RNA-cold RBP retrieval (MRR / Recall@K)
# ---------------------------------------------------------------------
RETR_KS = (5, 10, 20, 50)

def retrieval_metrics(scores, true_prots, ks=RETR_KS):
    order = np.argsort(-scores)
    ranks = {p: r for r, p in enumerate(order.tolist())}
    tp_ranks = sorted(ranks[p] for p in true_prots)
    out = {"MRR": 1.0 / (tp_ranks[0] + 1), "n_true": len(true_prots)}
    for k in ks:
        hits = sum(1 for r in tp_ranks if r < k)
        out[f"Recall@{k}"]    = hits / len(true_prots)
        out[f"Precision@{k}"] = hits / k
    return out

def run_rna_retrieval(model_factory, dataset, seed, n_splits=N_SPLITS):
    seed_all(seed)
    x_rna, x_prot, r_idx, p_idx, y, meta = build_graph_data(dataset)
    n_rna, n_prot = x_rna.shape[0], x_prot.shape[0]
    splits = build_rna_coldstart_splits(r_idx, p_idx, y, n_splits, seed)
    per_rna = []
    for fold, (tr_full, te) in enumerate(splits):
        tr, va = build_inner_val_split(r_idx, p_idx, y, tr_full,
                                       split_type="rna_cold", val_frac=0.10, seed=seed)
        model = model_factory()
        _, _, _, pack = train_one_fold(
            model, x_rna, x_prot, r_idx, p_idx, y,
            train_idx=tr, val_idx=va, test_idx=te,
            epochs=EPOCHS, lr=1e-3, bs=4096, early_stop_patience=5)
        rna_true = {}
        for e in te[y[te] == 1]:
            rna_true.setdefault(int(r_idx[e]), set()).add(int(p_idx[e]))
        query = [r for r, s in rna_true.items() if len(s) >= 1]
        if query:
            R, P = len(query), n_prot
            logits = get_edge_logits(model, **pack,
                                     r_idx_np=np.repeat(np.array(query), P),
                                     p_idx_np=np.tile(np.arange(P), R)).reshape(R, P)
            for i, r in enumerate(query):
                m = retrieval_metrics(logits[i], rna_true[r]); m.update(fold=fold, r=r)
                per_rna.append(m)
        del model, pack
        torch.cuda.empty_cache()
    return pd.DataFrame(per_rna)

def run_caseA():
    t0 = time.time()
    for dataset in DATASETS:
        for name, fac in CS_MODELS.items():
            for seed in SEEDS:
                fp = f"{OUTDIR}/A_raw_{dataset}_{name}_{seed}.csv"
                if os.path.exists(fp):
                    print(f"[skip] A {dataset:9s} {name:14s} seed={seed}")
                    continue
                print(f"[run ] A {dataset:9s} {name:14s} seed={seed}")
                df = run_rna_retrieval(fac, dataset, seed)
                df["dataset"], df["model"], df["seed"] = dataset, name, seed
                df.to_csv(fp, index=False)
    raw = pd.concat([pd.read_csv(f) for f in glob.glob(f"{OUTDIR}/A_raw_*.csv")],
                    ignore_index=True)
    raw.to_csv(f"{OUTDIR}/caseA_raw_perRNA.csv", index=False)
    print(f"Case A done in {(time.time()-t0)/60:.1f} min | query-RNAs={len(raw)}")
    return raw

retr_raw = run_caseA()

metric_cols = (["MRR"] + [f"Recall@{k}" for k in RETR_KS]
               + [f"Precision@{k}" for k in RETR_KS])
# per-seed mean over query RNAs -> aggregate over seeds
seed_means = retr_raw.groupby(["dataset", "model", "seed"])[metric_cols].mean().reset_index()
A_summary = (seed_means.groupby(["dataset", "model"])[metric_cols]
               .agg(["mean", "std"]))
A_summary.to_csv(f"{OUTDIR}/caseA_retrieval_summary.csv")
print("\n=== Case Study A: mean retrieval over query RNAs (mean over seeds) ===")
for dataset in DATASETS:
    print(f"\n--- {dataset} ---")
    print(seed_means[seed_means.dataset == dataset]
          .groupby("model")[metric_cols].mean().round(4).T)

# grouped bar of the discriminating metrics, with seed error bars
SHOW = ["MRR", "Recall@5", "Recall@10", "Recall@20"]
for dataset in DATASETS:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(SHOW)); w = 0.25
    sm = seed_means[seed_means.dataset == dataset]
    for i, mdl in enumerate(CS_MODELS):
        g = sm[sm.model == mdl]
        means = [g[c].mean() for c in SHOW]
        errs  = [g[c].std()  for c in SHOW]
        ax.bar(x + (i - 1) * w, means, w, yerr=errs, capsize=3,
               label=mdl, color=MODEL_COLORS[mdl])
    ax.set_xticks(x); ax.set_xticklabels(SHOW)
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.0)
    ax.set_title(f"RNA-cold RBP retrieval ({dataset})")
    ax.legend(); fig.tight_layout()
    fig.savefig(f"{OUTDIR}/figA_retrieval_{dataset}.png", dpi=200)
    plt.show()
